In [1]:
import copy
import math
import os
import random

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import RandomSampler
from torch.utils.data import Dataset, DataLoader, Subset, Sampler
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv


# TF32: ~2× faster matmul on Ampere+ GPUs; silently ignored on CPU.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

In [2]:
# ===================================================================
# MOORING LINE PROPERTIES  (Santjer et al. 2025 – Table 2)
# ===================================================================
MOORING_PROPERTIES = {
    "diameter":         0.068,        # [m]
    "area":             0.003619,     # [m2]
    "mass_per_length":  28.23,        # [kg/m]  (rhoA)
    "EA":               232.7e6,      # [N]     axial stiffness
    "Cd_n":             2.6,          # normal drag coefficient
    "Cd_t":             1.4,          # axial drag coefficient
    "k_b":              3.0e6,        # [Pa/m]  seabed spring stiffness
    "c_b":              3.0e5,        # [Pa*s/m] seabed damping
}


def build_mooring_graph(
    x0_nodes: torch.Tensor,
    z0_nodes: torch.Tensor,
    tension0_nodes: torch.Tensor,
    s0_nodes: torch.Tensor,
    water_depth: float,
    depths_current=None,
) -> Data:
    """
    Build a static PyG graph from the first time-step (t=0.1 s) of gnl_data1.

    Node positions (x0, z0) and initial tension come directly from the FE
    output at t=0.1 s, resampled to the desired node count.

    Static node features [N x 14]:
        0  s_over_L_line    normalised arc-length position [0..1]
        1  is_anchor        1 for r0
        2  is_fairlead      1 for rN
        3  is_intermediate  1 for interior nodes
        4  x0               initial x-coordinate [m]  (from gnl_data1 t=0)
        5  z0               initial z-coordinate [m]  (from gnl_data1 t=0)
        6  nodal_mass       lumped mass [kg]
        7  diameter         line diameter [m]
        8  area             cross-sectional area [m2]
        9  submerged_weight submerged weight contribution [N]
        10 z_bed            seabed elevation [m]  (= anchor z from gnl_data1)
        11 Cd_n             normal drag coefficient
        12 Cd_t             axial drag coefficient
        13 tension0         initial tension [N]  (from gnl_data1 t=0)

    Static edge features [E x 10]:
        0  segment_length             reference arc length [m]
        1  segment_length_over_L_line normalised segment length
        2  EA                         axial stiffness [N]
        3  mass_per_length            linear density [kg/m]
        4  diameter                   line diameter [m]
        5  tangent_x                  x-component of reference tangent
        6  tangent_z                  z-component of reference tangent
        7  edge_is_anchor               1.0 if anchor-side edge  else 0.0
        8  edge_is_intermediate         1.0 if internal edge     else 0.0
        9  edge_is_fairlead             1.0 if fairlead-side edge else 0.0

    Parameters
    ----------
    x0_nodes : torch.Tensor  shape [N]
        Initial x-coordinates from gnl_data1 row 0, resampled to N nodes.
    z0_nodes : torch.Tensor  shape [N]
        Initial z-coordinates from gnl_data1 row 0, resampled to N nodes.
    tension0_nodes : torch.Tensor  shape [N]
        Initial tension from gnl_data1 row 0, resampled to N nodes.
    s0_nodes : torch.Tensor  shape [N]
        Arc-length coordinate s [m] per node from gnl_data1 row 0, resampled to N nodes.
        s0_nodes[0] == 0 (anchor), s0_nodes[-1] == L_line (fairlead).
    water_depth : float
        Total water depth h0 [m] read from the input CSV for this case.
    depths_current : sequence, optional
        Current-profile depths d1..d5 [m] read from the input CSV for this case.
    """
    props = MOORING_PROPERTIES

    h = float(water_depth)
    if h <= 0.0:
        raise ValueError(f"water_depth must be positive, got {water_depth}.")


    # ------------------------------------------------------------------
    # 1. NODE POSITIONS  (from gnl_data1 first time step, t=0.1 s)
    # ------------------------------------------------------------------
    x0        = x0_nodes
    z0        = z0_nodes
    num_nodes = x0.shape[0]
    L_line    = float(s0_nodes[-1])   # actual arc length from gnl_data1

    # ------------------------------------------------------------------
    # 2. ARC-LENGTH COORDINATE along the mooring line
    # ------------------------------------------------------------------
    s_over_L = s0_nodes / L_line

    # ------------------------------------------------------------------
    # 3. NODE TYPE FLAGS
    # ------------------------------------------------------------------
    is_anchor       = torch.zeros(num_nodes)
    is_fairlead     = torch.zeros(num_nodes)
    is_intermediate = torch.ones(num_nodes)
    is_anchor[0]        = 1.0
    is_fairlead[-1]     = 1.0
    is_intermediate[0]  = 0.0
    is_intermediate[-1] = 0.0

    # ------------------------------------------------------------------
    # 4. PHYSICAL NODE ATTRIBUTES
    # ------------------------------------------------------------------
    seg_len_nom = L_line / (num_nodes - 1)

    nodal_mass = torch.full((num_nodes,), props["mass_per_length"] * seg_len_nom)
    nodal_mass[0]  *= 0.5
    nodal_mass[-1] *= 0.5

    diameter_node = torch.full((num_nodes,), props["diameter"])
    area_node     = torch.full((num_nodes,), props["area"])

    rho_water = 1025.0
    g         = 9.81
    submerged_weight_per_length = (
        props["mass_per_length"] - rho_water * props["area"]
    ) * g
    submerged_weight_node = torch.full(
        (num_nodes,), submerged_weight_per_length * seg_len_nom
    )
    submerged_weight_node[0]  *= 0.5
    submerged_weight_node[-1] *= 0.5

    z_bed      = float(z0[0])       # seabed level = anchor elevation
    z_bed_node = torch.full((num_nodes,), z_bed)

    Cd_n_node = torch.full((num_nodes,), props["Cd_n"])
    Cd_t_node = torch.full((num_nodes,), props["Cd_t"])

    # ------------------------------------------------------------------
    # 5. STATIC NODE FEATURE MATRIX  [N x 14]
    # ------------------------------------------------------------------
    x = torch.stack([
        s_over_L,              # 0
        is_anchor,             # 1
        is_fairlead,           # 2
        is_intermediate,       # 3
        x0,                    # 4
        z0,                    # 5
        nodal_mass,            # 6
        diameter_node,         # 7
        area_node,             # 8
        submerged_weight_node, # 9
        z_bed_node,            # 10
        Cd_n_node,             # 11
        Cd_t_node,             # 12
        tension0_nodes,        # 13
    ], dim=1)

    # ------------------------------------------------------------------
    # 6. GRAPH CONNECTIVITY  –  bidirectional chain
    # ------------------------------------------------------------------
    senders, receivers = [], []
    for i in range(num_nodes - 1):
        senders   += [i,     i + 1]
        receivers += [i + 1, i    ]
    edge_index = torch.tensor([senders, receivers], dtype=torch.long)

    # ------------------------------------------------------------------
    # 7. STATIC EDGE FEATURES  [E x 10]  (same for forward and backward)
    # ------------------------------------------------------------------
    edge_features = []
    for i in range(num_nodes - 1):
        dx = float(x0[i + 1] - x0[i])
        dz = float(z0[i + 1] - z0[i])
        seg_len    = math.sqrt(dx ** 2 + dz ** 2)
        seg_normed = seg_len / L_line
        tx = dx / seg_len if seg_len > 0 else 0.0
        tz = dz / seg_len if seg_len > 0 else 0.0
        edge_is_anchor       = 1.0 if i == 0             else 0.0
        edge_is_fairlead     = 1.0 if i == num_nodes - 2 else 0.0
        edge_is_intermediate = 1.0 if edge_is_anchor == 0.0 and edge_is_fairlead == 0.0 else 0.0
        feat = [
            seg_len,                   # 0
            seg_normed,                # 1
            props["EA"],               # 2
            props["mass_per_length"],  # 3
            props["diameter"],         # 4
            tx,                        # 5
            tz,                        # 6
            edge_is_anchor,            # 7
            edge_is_intermediate,      # 8
            edge_is_fairlead,          # 9
        ]
        edge_features.extend([feat, feat])    # forward + backward

    edge_attr = torch.tensor(edge_features, dtype=torch.float32)

    # ------------------------------------------------------------------
    # 8. GRAPH OBJECT
    # ------------------------------------------------------------------
    pos  = torch.stack([x0, z0], dim=1)
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, pos=pos)

    data.num_nodes_total  = num_nodes
    data.water_depth      = h
    data.line_length = L_line
    data.ea               = props["EA"]
    data.mass_per_length  = props["mass_per_length"]
    data.diameter         = props["diameter"]
    data.area             = props["area"]
    data.k_b              = props["k_b"]
    data.c_b              = props["c_b"]
    data.z_bed            = z_bed
    data.z_bed_node       = z_bed_node
    rope_positions        = [f"r{i}" for i in range(num_nodes)]
    data.rope_positions   = rope_positions
    data.depths_current   = None if depths_current is None else [float(d) for d in depths_current]

    return data


def print_graph_summary(data: Data):
    print("----- GRAPH SUMMARY -----")
    print(f"Water depth [m] : {data.water_depth:.1f}")
    print(f"L_line [m]      : {data.line_length:.2f}")
    print(f"EA [N]          : {data.ea:.3e}")
    print(f"Nodes           : {data.num_nodes_total}  ({', '.join(data.rope_positions)})")
    print()
    print(f"x shape       : {tuple(data.x.shape)}")
    print(f"edge_index    : {tuple(data.edge_index.shape)}")
    print(f"edge_attr     : {tuple(data.edge_attr.shape)}")
    print()
    print("Static node feature columns (14):")
    for i, c in enumerate([
        "s_over_L_line", "is_anchor", "is_fairlead", "is_intermediate",
        "x0", "z0", "nodal_mass", "diameter", "area",
        "submerged_weight", "z_bed", "Cd_n", "Cd_t", "tension0",
    ]):
        print(f"  {i:2d}  {c}")
    print()
    print("Static edge feature columns (10):")
    for i, c in enumerate([
        "segment_length", "segment_length_over_L_line", "EA",
        "mass_per_length", "diameter", "tangent_x", "tangent_z",
        "edge_is_anchor", "edge_is_intermediate", "edge_is_fairlead",
    ]):
        print(f"  {i}  {c}")
    print()
    print("Node positions (x, z) [m]:")
    print(data.pos)
    
# build_mooring_graph now requires x0_nodes, z0_nodes, tension0_nodes from gnl_data1.
# Call build_real_load_case_datasets() to construct graphs with real initial conditions.

In [3]:
# -------------------------------------------------------------
# resampling utility
# -------------------------------------------------------------



def resample_fe_output(data_tN: torch.Tensor, target_N: int) -> torch.Tensor:
    """
    Resample FE output from N_orig nodes to target_N nodes along the
    normalised arc-length axis using linear interpolation (np.interp).

    The anchor (node 0) and fairlead (node -1) are always preserved
    exactly because they are the endpoints of the np.linspace grid.

    Parameters
    ----------
    data_tN : torch.Tensor  shape [T, N_orig]
        Full FE output for a single signal (e.g. x_abs, z_abs, tension)
        at N_orig nodes.
    target_N : int
        Desired number of output nodes (includes anchor and fairlead).

    Returns
    -------
    torch.Tensor  shape [T, target_N], dtype=torch.float32
    """
    T, N_orig = data_tN.shape
    if target_N == N_orig:
        return data_tN.to(torch.float32)

    src_pos = np.linspace(0.0, 1.0, N_orig, dtype=np.float32)
    tgt_pos = np.linspace(0.0, 1.0, target_N, dtype=np.float32)

    # Find left-neighbour index for each target position
    idx = np.searchsorted(src_pos, tgt_pos, side="right") - 1
    idx = np.clip(idx, 0, N_orig - 2)                      # [target_N]

    # Linear interpolation weight in [0, 1]
    weight = (tgt_pos - src_pos[idx]) / (src_pos[idx + 1] - src_pos[idx])
    weight = np.clip(weight, 0.0, 1.0).astype(np.float32)  # [target_N]

    arr   = data_tN.numpy()         # zero-copy view of float32 tensor
    left  = arr[:, idx]             # [T, target_N]
    right = arr[:, idx + 1]         # [T, target_N]
    out   = left + weight * (right - left)  # [T, target_N]

    return torch.from_numpy(out.astype(np.float32))


In [4]:
ENV_COL_NAMES = [
    "h0", "Hs", "Tp",
    "d1", "v1",
    "d2", "v2",
    "d3", "v3",
    "d4", "v4",
    "d5", "v5",
]


class MooringSequenceDatasetPositionTension(Dataset):
    """
    Lazy sliding-window dataset for the mooring-line GAT-LSTM.

    Data is stored on disk as uncompressed .npy files and accessed via mmap.
    File handles are opened lazily inside each DataLoader worker on the first
    __getitem__ call, so 30,000 dataset objects consume negligible RAM.

    Dynamic node features (base 6 + 13 env = 19 total):
    ---------------------------------------------------------------
    0  x_abs             absolute horizontal node position [m]
    1  z_abs             absolute vertical node position [m]
    2  tension           line tension [N]
    3  contact_flag      binary: 1 if node touches seabed
    4  penetration_depth vertical penetration below seabed [m]
    5  bed_reaction_z    vertical seabed reaction force [N]
    6+ h0, Hs, Tp, d1, v1, ..., d5, v5  environmental features

    Target features (3):
    --------------------
    0  x_abs
    1  z_abs
    2  tension
    """

    def __init__(
        self,
        graph_data,
        case_dir: Path,
        target_N: int,
        history_len: int,
        future_len: int,
        contact_tol: float = 1e-6,
        max_time_steps: int = None,
    ):
        """
        Parameters
        ----------
        graph_data : torch_geometric.data.Data
            Static graph built by build_mooring_graph() for target_N nodes.
        case_dir : Path
            Directory containing x_abs.npy, z_abs.npy, tension.npy,
            reference.npy, env.npy for this (loc, case).
        target_N : int
            Number of nodes to resample to.
        history_len : int
        future_len : int
        contact_tol : float
            Tolerance for seabed contact detection [m].
        max_time_steps : int, optional
            If set, expose only the first max_time_steps rows (debug mode).
        """
        self.graph_data   = graph_data
        self._case_dir    = Path(case_dir)
        self._contact_tol = contact_tol
        self.target_N     = target_N
        self.history_len  = history_len
        self.future_len   = future_len

        # Read only the array shape — numpy reads just the .npy header
        T_full = np.load(self._case_dir / "x_abs.npy", mmap_mode="r").shape[0]
        self.num_steps = T_full if max_time_steps is None else min(T_full, max_time_steps)
        self.num_nodes = target_N
        self.num_windows = self.num_steps - history_len - future_len + 1

        if self.num_windows <= 0:
            raise ValueError(
                f"Not enough time steps ({self.num_steps}) for "
                f"history_len={history_len} + future_len={future_len}."
            )

        # Load the small env row eagerly (13 floats — negligible memory)
        self._env_row = torch.tensor(
            np.load(self._case_dir / "env.npy"), dtype=torch.float32
        )   # [13]

        # Feature-count metadata (used by build_model_from_dataset_example)
        self.n_dynamic_node_features = 6 + len(ENV_COL_NAMES)   # 19
        self.n_dynamic_edge_features = 4
        self.n_targets               = 3

        # Mmap handles — None until first __getitem__ in this worker process
        self._mmap = None

    # ------------------------------------------------------------------
    # Legacy attribute compatibility (used by print_dataset_summary and
    # build_model_from_dataset_example via .shape[-1])
    # ------------------------------------------------------------------
    @property
    def dynamic_feature_names(self):
        return [
            "x_abs", "z_abs", "tension", "contact_flag",
            "penetration_depth", "bed_reaction_z",
        ] + list(ENV_COL_NAMES)

    @property
    def dynamic_edge_feature_names(self):
        return [
            "edge_contact_fraction",
            "edge_penetration_mean",
            "edge_bed_reaction_mean",
            "edge_touchdown_flag",
        ]

    @property
    def target_feature_names(self):
        return ["x_abs", "z_abs", "tension"]

    def __len__(self):
        return self.num_windows

    def _open_mmap(self):
        """Open mmap handles for this worker process (called once per worker)."""
        self._mmap = {
            k: np.load(self._case_dir / f"{k}.npy", mmap_mode="r")
            for k in ("x_abs", "z_abs", "tension", "reference")
        }

    def __getitem__(self, idx):
        if idx < 0 or idx >= self.num_windows:
            raise IndexError(f"Window index {idx} out of range [0, {self.num_windows}).")

        # Open mmap lazily — each DataLoader worker opens its own file handles
        if self._mmap is None:
            self._open_mmap()

        start = idx
        end   = idx + self.history_len + self.future_len

        # 1. Read the required rows from mmap — OS page cache backed
        x_abs_w = torch.tensor(self._mmap["x_abs"][start:end],    dtype=torch.float32)
        z_abs_w = torch.tensor(self._mmap["z_abs"][start:end],    dtype=torch.float32)
        ten_w   = torch.tensor(self._mmap["tension"][start:end],   dtype=torch.float32)

        # 2. Resample from N_FULL=21 nodes to target_N
        x_abs   = resample_fe_output(x_abs_w, self.target_N)    # [win, target_N]
        z_abs   = resample_fe_output(z_abs_w, self.target_N)
        tension = resample_fe_output(ten_w,   self.target_N)

        # 3. Seabed interaction features
        z_bed          = self.graph_data.z_bed_node.detach().cpu().unsqueeze(0)   # [1, N]
        contact_flag   = (z_abs <= z_bed + self._contact_tol).float()
        penetration    = torch.clamp(z_bed - z_abs, min=0.0)
        submerged_w    = self.graph_data.x[:, 9].detach().cpu().unsqueeze(0)    # [1, N]
        bed_reaction   = torch.where(
            contact_flag > 0.0,
            submerged_w + self.graph_data.k_b * penetration,
            torch.zeros_like(penetration),
        )

        # 4. Dynamic node features [win_len, N, 19]
        win_len = end - start
        env_exp = self._env_row.unsqueeze(0).unsqueeze(0).expand(win_len, self.target_N, -1)
        dyn = torch.cat(
            [
                torch.stack(
                    [x_abs, z_abs, tension, contact_flag, penetration, bed_reaction],
                    dim=-1,
                ),   # [win, N, 6]
                env_exp,  # [win, N, 13]
            ],
            dim=-1,
        )   # [win, N, 19]

        # 5. Dynamic edge features [win_len, E, 4]
        src_nodes, tgt_nodes = self.graph_data.edge_index
        edge_dyn = torch.stack(
            [
                0.5 * (contact_flag[:, src_nodes] + contact_flag[:, tgt_nodes]),
                0.5 * (penetration[:, src_nodes]  + penetration[:, tgt_nodes]),
                0.5 * (bed_reaction[:, src_nodes]  + bed_reaction[:, tgt_nodes]),
                (contact_flag[:, src_nodes] != contact_flag[:, tgt_nodes]).float(),
            ],
            dim=-1,
        )   # [win, E, 4]

        h = self.history_len
        return {
            "graph":     self.graph_data.clone(),
            "x_seq":     dyn[:h],                                                     # [H, N, 19]
            "edge_seq":  edge_dyn[:h],                                                # [H, E, 4]
            "y_seq":     torch.stack([x_abs[h:], z_abs[h:], tension[h:]], dim=-1),   # [P, N, 3]
            "start_idx": idx,
        }


def print_dataset_summary(dataset: MooringSequenceDatasetPositionTension):
    print("----- DATASET SUMMARY -----")
    print(f"Time steps : {dataset.num_steps}")
    print(f"Nodes      : {dataset.num_nodes}")
    print(f"History len: {dataset.history_len}")
    print(f"Future len : {dataset.future_len}")
    print(f"Windows    : {len(dataset)}")
    print("Dynamic input features :", dataset.dynamic_feature_names)
    print("Dynamic edge  features :", dataset.dynamic_edge_feature_names)
    print("Target features        :", dataset.target_feature_names)
    sample = dataset[0]
    print("x_seq   :", tuple(sample["x_seq"].shape))
    print("edge_seq:", tuple(sample["edge_seq"].shape))
    print("y_seq   :", tuple(sample["y_seq"].shape))


In [5]:
class MooringGATEncoder(nn.Module):
    """
    Spatial graph encoder for one history time step.

    It takes:
      - static node features from graph.x
      - static edge features from graph.edge_attr
      - dynamic node features for one time step x_t
      - dynamic edge features for one time step edge_t

    and returns:
      - node embeddings h_t for that time step

    Expected shapes for one sample:
      graph.x         : [N, n_static_node_features]
      graph.edge_index: [2, E]
      graph.edge_attr : [E, n_static_edge_features]
      x_t             : [N, n_dynamic_node_features]
      edge_t          : [E, n_dynamic_edge_features]

    Output:
      h_t             : [N, gat_hidden_dim]
    """

    def __init__(
        self,
        n_static_node_features: int = 14,
        n_dynamic_node_features: int = 6,
        n_static_edge_features: int = 10,
        n_dynamic_edge_features: int = 4,
        gat_hidden_dim: int = 64,
        gat_out_dim: int = 64,
        num_heads: int = 4,
        dropout: float = 0.1,
        use_layernorm: bool = True,
        add_residual_projection: bool = True,
    ):
        super().__init__()

        self.n_static_node_features = n_static_node_features
        self.n_dynamic_node_features = n_dynamic_node_features
        self.n_static_edge_features = n_static_edge_features
        self.n_dynamic_edge_features = n_dynamic_edge_features

        self.node_input_dim = n_static_node_features + n_dynamic_node_features
        self.edge_input_dim = n_static_edge_features + n_dynamic_edge_features

        self.gat_hidden_dim = gat_hidden_dim
        self.gat_out_dim = gat_out_dim
        self.num_heads = num_heads
        self.dropout = dropout
        self.use_layernorm = use_layernorm

        # -------------------------------------------------------------
        # GAT layer 1
        # concat=True => output dim = gat_hidden_dim * num_heads
        # -------------------------------------------------------------
        self.gat1 = GATv2Conv(
            in_channels=self.node_input_dim,
            out_channels=gat_hidden_dim,
            heads=num_heads,
            concat=True,
            dropout=dropout,
            edge_dim=self.edge_input_dim,
            add_self_loops=False,
            bias=True,
        )

        self.norm1 = nn.LayerNorm(gat_hidden_dim * num_heads) if use_layernorm else nn.Identity()

        # -------------------------------------------------------------
        # GAT layer 2
        # concat=False => output dim = gat_out_dim
        # -------------------------------------------------------------
        self.gat2 = GATv2Conv(
            in_channels=gat_hidden_dim * num_heads,
            out_channels=gat_out_dim,
            heads=1,
            concat=False,
            dropout=dropout,
            edge_dim=self.edge_input_dim,
            add_self_loops=False,
            bias=True,
        )

        self.norm2 = nn.LayerNorm(gat_out_dim) if use_layernorm else nn.Identity()

        self.act = nn.ELU()
        self.dropout_layer = nn.Dropout(dropout)

        # Optional residual projection from raw concatenated node input
        if add_residual_projection:
            self.residual_proj = nn.Linear(self.node_input_dim, gat_out_dim)
        else:
            self.residual_proj = None

    def forward(self, graph, x_t: torch.Tensor, edge_t: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        graph : torch_geometric.data.Data
            Static graph object containing graph.x, graph.edge_index, graph.edge_attr.
        x_t : torch.Tensor
            Dynamic node features at one time step, shape [N, n_dynamic_node_features].
        edge_t : torch.Tensor
            Dynamic edge features at one time step, shape [E, n_dynamic_edge_features].

        Returns
        -------
        h_t : torch.Tensor
            Encoded node embeddings for this time step, shape [N, gat_out_dim].
        """

        # -------------------------------------------------------------
        # Basic shape checks
        # -------------------------------------------------------------
        if x_t.dim() != 2:
            raise ValueError(f"x_t must have shape [N, F_dyn_node], got {tuple(x_t.shape)}")

        if edge_t.dim() != 2:
            raise ValueError(f"edge_t must have shape [E, F_dyn_edge], got {tuple(edge_t.shape)}")

        x_static = graph.x
        edge_index = graph.edge_index
        edge_static = graph.edge_attr

        if x_static.size(0) != x_t.size(0):
            raise ValueError(
                f"Node count mismatch: graph.x has {x_static.size(0)} nodes, "
                f"but x_t has {x_t.size(0)} nodes."
            )

        if edge_static.size(0) != edge_t.size(0):
            raise ValueError(
                f"Edge count mismatch: graph.edge_attr has {edge_static.size(0)} edges, "
                f"but edge_t has {edge_t.size(0)} edges."
            )

        if x_static.size(1) != self.n_static_node_features:
            raise ValueError(
                f"Expected {self.n_static_node_features} static node features, "
                f"got {x_static.size(1)}."
            )

        if x_t.size(1) != self.n_dynamic_node_features:
            raise ValueError(
                f"Expected {self.n_dynamic_node_features} dynamic node features, "
                f"got {x_t.size(1)}."
            )

        if edge_static.size(1) != self.n_static_edge_features:
            raise ValueError(
                f"Expected {self.n_static_edge_features} static edge features, "
                f"got {edge_static.size(1)}."
            )

        if edge_t.size(1) != self.n_dynamic_edge_features:
            raise ValueError(
                f"Expected {self.n_dynamic_edge_features} dynamic edge features, "
                f"got {edge_t.size(1)}."
            )

        # -------------------------------------------------------------
        # Concatenate static + dynamic features for this time step
        # -------------------------------------------------------------
        x_in = torch.cat([x_static, x_t], dim=-1)           
        edge_in = torch.cat([edge_static, edge_t], dim=-1) 

        # -------------------------------------------------------------
        # GAT block 1
        # -------------------------------------------------------------
        h = self.gat1(x_in, edge_index, edge_in)            # [N, gat_hidden_dim * num_heads]
        h = self.norm1(h)
        h = self.act(h)
        h = self.dropout_layer(h)

        # -------------------------------------------------------------
        # GAT block 2
        # -------------------------------------------------------------
        h = self.gat2(h, edge_index, edge_in)               # [N, gat_out_dim]
        h = self.norm2(h)

        # -------------------------------------------------------------
        # Optional residual connection from raw input
        # -------------------------------------------------------------
        if self.residual_proj is not None:
            h = h + self.residual_proj(x_in)

        h = self.act(h)
        h = self.dropout_layer(h)

        return h

    def forward_batch(
        self,
        x_static: torch.Tensor,
        x_dynamic: torch.Tensor,
        edge_static: torch.Tensor,
        edge_dynamic: torch.Tensor,
        edge_index: torch.Tensor,
    ) -> torch.Tensor:
        """
        Vectorised forward over M independent graphs with shared topology.
        Builds one block-diagonal super-graph (M*N nodes, M*E edges) and
        runs GATv2Conv once instead of M serial kernel launches.

        Parameters
        ----------
        x_static    : [M, N, F_static_node]
        x_dynamic   : [M, N, F_dyn_node]
        edge_static : [M, E, F_static_edge]
        edge_dynamic: [M, E, F_dyn_edge]
        edge_index  : [2, E]  shared topology for all M graphs
        """
        M, N, _ = x_dynamic.shape
        E = edge_index.size(1)
        device = x_dynamic.device

        x_in    = torch.cat([x_static,    x_dynamic],   dim=-1)  # [M, N, F_node_total]
        edge_in = torch.cat([edge_static, edge_dynamic], dim=-1)  # [M, E, F_edge_total]

        x_in_flat    = x_in.view(M * N, -1)    # [M*N, F_node_total]
        edge_in_flat = edge_in.view(M * E, -1) # [M*E, F_edge_total]

        # Tile edge_index with per-graph node offsets -> block-diagonal adjacency
        offsets  = torch.arange(M, device=device) * N               # [M]
        ei_tiled = edge_index.unsqueeze(1) + offsets.view(1, M, 1)  # [2, M, E]
        ei_flat  = ei_tiled.reshape(2, M * E)                       # [2, M*E]

        h = self.gat1(x_in_flat, ei_flat, edge_in_flat)
        h = self.norm1(h)
        h = self.act(h)
        h = self.dropout_layer(h)

        h = self.gat2(h, ei_flat, edge_in_flat)
        h = self.norm2(h)

        if self.residual_proj is not None:
            h = h + self.residual_proj(x_in_flat)

        h = self.act(h)
        h = self.dropout_layer(h)

        return h.view(M, N, self.gat_out_dim)  # [M, N, gat_out_dim]


In [6]:
class NodeTemporalLSTM(nn.Module):
    """
    Temporal encoder that processes the sequence of spatial node embeddings
    produced by the MooringGATEncoder.

    Input:
        H : [history_len, N, gat_out_dim]

    Output:
        node_temporal : [N, lstm_hidden_dim]

    Interpretation:
        For each node i, we take its embedding sequence across time:
            H[:, i, :]  -> [history_len, gat_out_dim]
        and pass it through an LSTM.

    Notes:
        - This block is node-wise in time.
        - It does NOT mix nodes with each other.
        - Spatial coupling has already been handled by the GAT encoder.
    """

    def __init__(
        self,
        input_dim: int = 64,
        lstm_hidden_dim: int = 128,
        num_lstm_layers: int = 1,
        dropout: float = 0.1,
        bidirectional: bool = False,
        use_layernorm: bool = True,
        use_last_timestep: bool = True,
    ):
        super().__init__()

        self.input_dim = input_dim
        self.lstm_hidden_dim = lstm_hidden_dim
        self.num_lstm_layers = num_lstm_layers
        self.bidirectional = bidirectional
        self.use_layernorm = use_layernorm
        self.use_last_timestep = use_last_timestep

        # PyTorch LSTM only uses dropout internally when num_layers > 1
        lstm_dropout = dropout if num_lstm_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=num_lstm_layers,
            batch_first=True,   
            dropout=lstm_dropout,
            bidirectional=bidirectional,
        )

        self.output_dim = lstm_hidden_dim * (2 if bidirectional else 1)

        self.norm = nn.LayerNorm(self.output_dim) if use_layernorm else nn.Identity()
        self.dropout_layer = nn.Dropout(dropout)

    def forward(self, H: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        H : torch.Tensor
            Sequence of node embeddings from the spatial encoder.
            Expected shape: [history_len, N, input_dim]

        Returns
        -------
        node_temporal : torch.Tensor
            Temporal embedding for each node.
            Shape: [N, output_dim]
        """

        if H.dim() != 3:
            raise ValueError(
                f"H must have shape [history_len, N, input_dim], got {tuple(H.shape)}"
            )

        history_len, N, F = H.shape

        if F != self.input_dim:
            raise ValueError(
                f"Expected input_dim={self.input_dim}, but got last dimension {F}."
            )

        # Rearrange so each node becomes one sequence sample:
        # [history_len, N, input_dim] -> [N, history_len, input_dim]
        H_nodes = H.permute(1, 0, 2).contiguous()

        # LSTM output:
        #   lstm_out : [N, history_len, output_dim]
        #   h_n      : [num_layers * num_directions, N, lstm_hidden_dim]
        lstm_out, (h_n, c_n) = self.lstm(H_nodes)

        if self.use_last_timestep:
            # Use the output at the final history step
            node_temporal = lstm_out[:, -1, :]   # [N, output_dim]
        else:
            # Alternative: use the final hidden state
            if self.bidirectional:
                # last layer forward + last layer backward
                h_forward = h_n[-2]   # [N, lstm_hidden_dim]
                h_backward = h_n[-1]  # [N, lstm_hidden_dim]
                node_temporal = torch.cat([h_forward, h_backward], dim=-1)
            else:
                node_temporal = h_n[-1]  # [N, lstm_hidden_dim]

        node_temporal = self.norm(node_temporal)
        node_temporal = self.dropout_layer(node_temporal)

        return node_temporal


In [7]:
class MooringGATLSTM(nn.Module):
    """
    Full spatiotemporal model for the mooring-line problem.

    Pipeline:
        1) For each history time step t:
              - combine static + dynamic graph information
              - run MooringGATEncoder
              - get node embeddings h_t

        2) Stack all h_t over the history window:
              H = [history_len, N, gat_out_dim]

        3) Run NodeTemporalLSTM on H:
              node_temporal = [N, temporal_dim]

        4) Predict all future steps directly for each node:
              y_hat = [future_len, N, output_dim]

    Default output_dim=3 corresponds to:
        [x_abs, z_abs, tension]
    """

    def __init__(
        self,
        # ----- graph feature sizes -----
        n_static_node_features: int = 14,
        n_dynamic_node_features: int = 6,
        n_static_edge_features: int = 10,
        n_dynamic_edge_features: int = 4,

        # ----- spatial encoder -----
        gat_hidden_dim: int = 64,
        gat_out_dim: int = 64,
        num_heads: int = 4,
        gat_dropout: float = 0.1,
        gat_use_layernorm: bool = True,
        add_residual_projection: bool = True,

        # ----- temporal encoder -----
        lstm_hidden_dim: int = 128,
        num_lstm_layers: int = 1,
        lstm_dropout: float = 0.1,
        bidirectional: bool = False,
        lstm_use_layernorm: bool = True,
        use_last_timestep: bool = True,

        # ----- prediction head -----
        future_len: int = 1,
        output_dim: int = 3,
        head_hidden_dim: int = 128,
        head_dropout: float = 0.1,
    ):
        super().__init__()

        self.future_len = future_len
        self.output_dim = output_dim

        # -------------------------------------------------------------
        # Spatial encoder: one time step -> node embeddings
        # -------------------------------------------------------------
        self.spatial_encoder = MooringGATEncoder(
            n_static_node_features=n_static_node_features,
            n_dynamic_node_features=n_dynamic_node_features,
            n_static_edge_features=n_static_edge_features,
            n_dynamic_edge_features=n_dynamic_edge_features,
            gat_hidden_dim=gat_hidden_dim,
            gat_out_dim=gat_out_dim,
            num_heads=num_heads,
            dropout=gat_dropout,
            use_layernorm=gat_use_layernorm,
            add_residual_projection=add_residual_projection,
        )

        # -------------------------------------------------------------
        # Temporal encoder: sequence of node embeddings -> one temporal
        # embedding per node
        # -------------------------------------------------------------
        self.temporal_encoder = NodeTemporalLSTM(
            input_dim=gat_out_dim,
            lstm_hidden_dim=lstm_hidden_dim,
            num_lstm_layers=num_lstm_layers,
            dropout=lstm_dropout,
            bidirectional=bidirectional,
            use_layernorm=lstm_use_layernorm,
            use_last_timestep=use_last_timestep,
        )

        temporal_out_dim = self.temporal_encoder.output_dim

        # -------------------------------------------------------------
        # Prediction head:
        # [N, temporal_out_dim] -> [N, future_len * output_dim]
        # then reshape to [future_len, N, output_dim]
        # -------------------------------------------------------------
        self.prediction_head = nn.Sequential(
            nn.Linear(temporal_out_dim, head_hidden_dim),
            nn.ELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_hidden_dim, future_len * output_dim),
        )

        # -------------------------------------------------------------
        # Stage C2 (Kendall): learnable homoscedastic uncertainties for
        # the 4 physical loss terms. log_sigma[i] = log(sigma_i^2); the
        # loss combines each term as 0.5*exp(-s_i)*L_i + 0.5*s_i.
        # Initialised at 0 -> sigma^2 = 1 (every term starts unit-weighted).
        # Inert unless cfg.use_kendall_weighting=True.
        # -------------------------------------------------------------
        self.log_sigma = nn.Parameter(torch.zeros(4))

    def forward(self, graph_or_graphs, x_seq: torch.Tensor, edge_seq: torch.Tensor) -> torch.Tensor:
        """
        Dispatches to _forward_single based on input dimensionality.

        Unbatched  — graph_or_graphs: Data,
                     x_seq: [H, N, F_node], edge_seq: [H, E, F_edge]
                     returns [P, N, output_dim]

        Batched    — graph_or_graphs: List[Data] (all share same N, E),
                     x_seq: [B, H, N, F_node], edge_seq: [B, H, E, F_edge]
                     returns [B, P, N, output_dim]

        The batched path loops over B (correctness-first; vectorization deferred).
        Bucketed batching guarantees all B samples share the same N and E.
        """
        if x_seq.dim() == 4:
            return self._forward_batched(graph_or_graphs, x_seq, edge_seq)
        return self._forward_single(graph_or_graphs, x_seq, edge_seq)

    def _forward_single(self, graph, x_seq: torch.Tensor, edge_seq: torch.Tensor) -> torch.Tensor:
        """
        Single-sample forward pass.
        Parameters
        ----------
        graph : torch_geometric.data.Data
            Static graph object.
        x_seq : torch.Tensor
            Dynamic node features over the history window.
            Expected shape: [history_len, N, n_dynamic_node_features]
        edge_seq : torch.Tensor
            Dynamic edge features over the history window.
            Expected shape: [history_len, E, n_dynamic_edge_features]

        Returns
        -------
        y_hat : torch.Tensor
            Predicted future targets.
            Shape: [future_len, N, output_dim]
        """

        # -------------------------------------------------------------
        # Basic shape checks
        # -------------------------------------------------------------
        if x_seq.dim() != 3:
            raise ValueError(
                f"x_seq must have shape [history_len, N, F_dyn_node], got {tuple(x_seq.shape)}"
            )

        if edge_seq.dim() != 3:
            raise ValueError(
                f"edge_seq must have shape [history_len, E, F_dyn_edge], got {tuple(edge_seq.shape)}"
            )

        history_len_x, N_x, _ = x_seq.shape
        history_len_e, E_x, _ = edge_seq.shape

        if history_len_x != history_len_e:
            raise ValueError(
                f"History length mismatch: x_seq has {history_len_x}, edge_seq has {history_len_e}."
            )

        if graph.x.size(0) != N_x:
            raise ValueError(
                f"Node count mismatch: graph.x has {graph.x.size(0)} nodes, but x_seq has {N_x}."
            )

        if graph.edge_attr.size(0) != E_x:
            raise ValueError(
                f"Edge count mismatch: graph.edge_attr has {graph.edge_attr.size(0)} edges, "
                f"but edge_seq has {E_x}."
            )

        # Vectorised: one GATv2Conv call over H*N nodes (block-diagonal super-graph)
        # replaces the serial Python loop that launched H separate kernel calls.
        x_static  = graph.x.unsqueeze(0).expand(history_len_x, N_x, -1)
        ea_static = graph.edge_attr.unsqueeze(0).expand(history_len_x, E_x, -1)
        H = self.spatial_encoder.forward_batch(
            x_static, x_seq, ea_static, edge_seq, graph.edge_index
        )  # [H, N, gat_out_dim]

        node_temporal = self.temporal_encoder(H)                     # [N, temporal_out_dim]
        y_hat_flat    = self.prediction_head(node_temporal)          # [N, future_len * output_dim]
        y_hat = y_hat_flat.view(N_x, self.future_len, self.output_dim)
        return y_hat.permute(1, 0, 2).contiguous()                  # [P, N, output_dim]

    def _forward_batched(
        self,
        graphs,
        x_seq: torch.Tensor,
        edge_seq: torch.Tensor,
    ) -> torch.Tensor:
        """
        Batched forward, vectorised over B*H graphs in one GATv2Conv call.
        All B graphs share the same edge_index (same-N mooring chains);
        static node/edge features may differ per sample.
        """
        B, H, N, _ = x_seq.shape
        E = edge_seq.size(2)

        # Stack per-sample static features, expand over H, flatten to [B*H, ...]
        x_static_B  = torch.stack([g.x        for g in graphs], dim=0)  # [B, N, F_static]
        ea_static_B = torch.stack([g.edge_attr for g in graphs], dim=0) # [B, E, F_static_edge]
        x_static_flat  = x_static_B.unsqueeze(1).expand(B, H, N, -1).contiguous().view(B * H, N, -1)
        ea_static_flat = ea_static_B.unsqueeze(1).expand(B, H, E, -1).contiguous().view(B * H, E, -1)

        # Single GATv2Conv call for all B*H graphs
        H_enc_flat = self.spatial_encoder.forward_batch(
            x_static_flat,
            x_seq.view(B * H, N, -1),
            ea_static_flat,
            edge_seq.view(B * H, E, -1),
            graphs[0].edge_index,  # shared topology for same-N chain graphs
        )  # [B*H, N, gat_out_dim]

        H_enc = H_enc_flat.view(B, H, N, -1)  # [B, H, N, gat_out_dim]

        # Vectorise LSTM and prediction head over all B*N node sequences at once.
        # LSTM has no cross-sample interactions so results are identical to the B-loop.
        # [B, H, N, F] -> permute -> [B, N, H, F] -> reshape -> [B*N, H, F]
        gat_out_dim = H_enc.shape[-1]
        H_lstm = H_enc.permute(0, 2, 1, 3).reshape(B * N, H, gat_out_dim)  # [B*N, H, F]

        lstm_out, _ = self.temporal_encoder.lstm(H_lstm)     # [B*N, H, hidden]
        node_flat = lstm_out[:, -1, :]                        # [B*N, hidden]
        node_flat = self.temporal_encoder.norm(node_flat)
        node_flat = self.temporal_encoder.dropout_layer(node_flat)

        y_hat_flat = self.prediction_head(node_flat)          # [B*N, future_len * output_dim]
        y_hat = y_hat_flat.view(B, N, self.future_len, self.output_dim)
        return y_hat.permute(0, 2, 1, 3).contiguous()        # [B, P, N, output_dim]


In [ ]:
# -------------------------------------------------------------
# training config
# -------------------------------------------------------------

@dataclass
class TrainingConfig:
    split_seed: int = 42
    seed_list: Tuple[int, ...] = (42,)

    # Locations available from batchRieke dataset (loc02 and loc12 not used)
    # Locs 1,3-9 used for training (time-based train/val/test split per case)
    # Locs 10,11 used as held-out location-level test set
    train_lc_ids:      Tuple[int, ...] = (1, 3, 4, 5, 6, 7, 8, 9)
    test_extra_lc_ids: Tuple[int, ...] = (10, 11)
    node_counts: Tuple[int, ...] = (4, 5, 6, 7, 8, 10, 12, 15, 18, 21)

    test_time_fraction: float = 0.30
    train_fraction_within_dev_blocks: float = 0.70
    raw_block_len: int = 350


    # optimization
    num_epochs: int = 2
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    batch_size: int = 1
    grad_clip_max_norm: float = 1.0

    # scheduler / stopping
    use_scheduler: bool = False
    scheduler_factor: float = 0.5
    scheduler_patience: int = 5
    early_stopping_patience: int = 2
    min_delta: float = 1e-5

    checkpoint_dir: str = "./checkpoints_gat_lstm_debug"
    eps: float = 1e-8

    # Optional caps for first real-data debug runs. Set to None for full experiments.
    max_train_windows: Optional[int] = 256
    max_val_windows: Optional[int] = 128
    max_test_windows: Optional[int] = 128

    # Dynamic node feature indices ? layout (6 base + 13 env = 19 total):
    #   0  x_abs             continuous
    #   1  z_abs             continuous
    #   2  tension           continuous
    #   3  contact_flag      binary  (excluded from normalisation)
    #   4  penetration_depth continuous
    #   5  bed_reaction_z    continuous
    #   6  h0                continuous  (env water depth from CSV)
    #   7  Hs                continuous
    #   8  Tp                continuous
    #   9  d1                continuous
    #  10  v1                continuous
    #  11  d2                continuous
    #  12  v2                continuous
    #  13  d3                continuous
    #  14  v3                continuous
    #  15  d4                continuous
    #  16  v4                continuous
    #  17  d5                continuous
    #  18  v5                continuous
    node_continuous_idx: Tuple[int, ...] = (0, 1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18)
    node_binary_idx:     Tuple[int, ...] = (3,)

    edge_continuous_idx: Tuple[int, ...] = (0, 1, 2)
    edge_binary_idx:     Tuple[int, ...] = (3,)

    target_continuous_idx: Tuple[int, ...] = (0, 1, 2)

    # Full-scale data pipeline settings
    # num_workers=0 keeps single-process loading (safe on Windows with mmap)
    # Set num_workers=4 when running on Linux or after verifying Windows compat
    num_workers:                 int           = 0
    pin_memory:                  bool          = False
    max_train_windows_per_epoch: Optional[int] = 50_000   # None = use all windows
    n_fit_windows:               int           = 2_000    # windows for normalizer fitting

    # --- AMP & precision -------------------------------------------------
    use_amp:             bool          = False   # fp16 on CUDA, no-op on CPU

    # --- Training efficiency ---------------------------------------------
    validate_every:      int           = 1      # run val every N epochs; always on last epoch
    accumulation_steps:  int           = 1      # gradient accumulation steps
    save_every_n_epochs: int           = 0      # 0 = disabled; periodic checkpoint every N epochs

    # --- Model architecture (passed to build_model_from_config) ----------
    gat_hidden_dim:  int   = 64
    gat_out_dim:     int   = 64
    num_heads:       int   = 4
    gat_dropout:     float = 0.1
    lstm_hidden_dim: int   = 128
    num_lstm_layers: int   = 1
    lstm_dropout:    float = 0.1
    head_hidden_dim: int   = 128
    head_dropout:    float = 0.1

    # --- Metrics ---------------------------------------------------------
    tension_mape_min: float = 1000.0   # ignore MAPE where |true_tension| < this [N]

    # --- Checkpoint resume -----------------------------------------------
    resume_from_checkpoint: Optional[str] = None

    # --- Batching -------------------------------------------------------
    enable_bucketed_node_count_batching: bool = False
    drop_last_batch:                     bool = False

    # --- Debug ----------------------------------------------------------
    debug_batch_shapes:                  bool = False

    # --- Evaluation gating -------------------------------------------
    run_test_evaluation:                 bool = False  # set True only for final Stage C runs

    # --- Training physical metrics (optional, capped subset of train batches) -
    compute_train_physical_metrics: bool = False
    max_train_metric_batches: int = 100

    # --- Physical loss terms (Stage 2, all disabled by default) ---------------
    use_physical_loss_terms: bool = False
    w_phys_mae: float = 0.0
    w_peak_tension: float = 0.0
    w_peak_underprediction: float = 0.0
    w_temporal_smoothness: float = 0.0
    # --- Stage C2: Kendall multi-task uncertainty weighting -------------------
    # When True, the 4 w_* above are ignored and 4 learnable log_sigma params
    # (on MooringGATLSTM) weight the physical terms instead.
    use_kendall_weighting: bool = False
    # None = use target_norm.std[i] (recommended; auto-calibrated from training data)
    phys_x_scale: Optional[float] = None
    phys_z_scale: Optional[float] = None
    phys_tension_scale: Optional[float] = None

    # --- Composite checkpoint selection (disabled by default) -----------------
    use_composite_checkpoint_score: bool = False
    checkpoint_alpha: float = 0.0   # weight on normalised val_peak_tension_MAE
    checkpoint_beta: float = 0.0    # weight on normalised peak_underpred_rate
    # C2b: select best checkpoint by max val_global_R2_tension (not val_loss).
    select_on_r2_tension: bool = False
    # C2c: tau-quantile (pinball) for the peak-underprediction term so its sigma
    # is Kendall-rankable (two-sided + noise floor). tau=0.9 penalises under 9x over.
    peak_pinball_tau: float = 0.9


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


In [9]:
# -------------------------------------------------------------
# split utilities
# -------------------------------------------------------------

def get_window_span(dataset: MooringSequenceDatasetPositionTension) -> int:
    return dataset.history_len + dataset.future_len

def valid_window_starts_inside_raw_interval(
    dataset: MooringSequenceDatasetPositionTension,
    raw_start: int,
    raw_end_exclusive: int,
) -> List[int]:
    span = dataset.history_len + dataset.future_len
    lo = max(raw_start, 0)
    hi = min(raw_end_exclusive - span + 1, len(dataset))
    return list(range(lo, hi))

def chunk_raw_time_range(
    raw_start: int,
    raw_end_exclusive: int,
    block_len: int,
) -> List[Tuple[int, int]]:
    blocks = []
    cur = raw_start
    while cur < raw_end_exclusive:
        nxt = min(cur + block_len, raw_end_exclusive)
        blocks.append((cur, nxt))
        cur = nxt
    return blocks

def _sample_windows(starts, max_windows, rng):
    """Randomly subsample starts to at most max_windows entries.

    If max_windows is None or len(starts) <= max_windows, returns starts unchanged.
    Uses random.sample so the result is a new list (no mutation).
    """
    if max_windows is None or len(starts) <= max_windows:
        return starts
    return rng.sample(starts, max_windows)


def build_leakage_aware_splits(
    load_case_datasets: Dict[Tuple, MooringSequenceDatasetPositionTension],
    cfg: TrainingConfig,
) -> Dict[str, List]:
    """
    Build train / val / test window index lists.

    Dataset keys are (lc_id, case_id, target_N) 3-tuples.

    Split rules:
    - lc_id in train_lc_ids:
        * last 30% raw time per case -> test
        * first 70% raw time per case -> development (block-shuffled train/val)
    - lc_id in test_extra_lc_ids:
        * all windows -> test  (held-out locations)

    Each element of the returned lists is ((lc_id, case_id, target_N), window_idx).
    """
    rng = random.Random(cfg.split_seed)
    split = {"train": [], "val": [], "test": []}

    # -- Per-dataset window budgets (cap-aware, stratified) ---------------
    # Derive actual dataset counts from the dict keys; no dependency on globals.
    # Per-dataset budgets create stratified sampling: each dataset contributes
    # proportionally. Reproducibility depends on cfg.split_seed.
    _train_keys = [k for k in load_case_datasets if k[0] in cfg.train_lc_ids]
    _test_keys  = [k for k in load_case_datasets if k[0] in cfg.test_extra_lc_ids]
    _n_train_ds = max(1, len(_train_keys))
    _n_test_ds  = max(1, len(_test_keys))

    # Each training dataset contributes to both train and val; budget per dataset.
    _per_ds_train = (
        max(1, cfg.max_train_windows // _n_train_ds) if cfg.max_train_windows else None
    )
    _per_ds_val = (
        max(1, cfg.max_val_windows // _n_train_ds) if cfg.max_val_windows else None
    )
    _per_ds_test_held = (
        max(1, cfg.max_test_windows // _n_test_ds) if cfg.max_test_windows else None
    )
    # Time-based test windows from training locations share the same cap pool
    _per_ds_test_time = (
        max(1, cfg.max_test_windows // _n_train_ds) if cfg.max_test_windows else None
    )

    for (lc_id, case_id, target_N), ds in load_case_datasets.items():
        num_steps = ds.num_steps

        if lc_id in cfg.train_lc_ids:
            test_raw_start = int(math.floor((1.0 - cfg.test_time_fraction) * num_steps))

            test_window_ids = valid_window_starts_inside_raw_interval(
                ds, raw_start=test_raw_start, raw_end_exclusive=num_steps,
            )
            test_window_ids = _sample_windows(test_window_ids, _per_ds_test_time, rng)
            split["test"].extend(
                ((lc_id, case_id, target_N), w) for w in test_window_ids
            )

            blocks = chunk_raw_time_range(
                raw_start=0, raw_end_exclusive=test_raw_start,
                block_len=cfg.raw_block_len,
            )
            valid_blocks = []
            for b_start, b_end in blocks:
                w_ids = valid_window_starts_inside_raw_interval(
                    ds, raw_start=b_start, raw_end_exclusive=b_end,
                )
                if w_ids:
                    valid_blocks.append((b_start, b_end, w_ids))

            rng.shuffle(valid_blocks)
            n_train = int(math.floor(cfg.train_fraction_within_dev_blocks * len(valid_blocks)))

            # Flatten all block windows per dataset, then sample once (true per-dataset cap)
            train_candidates = [w for _, _, ww in valid_blocks[:n_train] for w in ww]
            val_candidates   = [w for _, _, ww in valid_blocks[n_train:] for w in ww]
            split["train"].extend(
                ((lc_id, case_id, target_N), w)
                for w in _sample_windows(train_candidates, _per_ds_train, rng)
            )
            split["val"].extend(
                ((lc_id, case_id, target_N), w)
                for w in _sample_windows(val_candidates, _per_ds_val, rng)
            )

        elif lc_id in cfg.test_extra_lc_ids:
            all_wins = list(range(len(ds)))   # range -> list for random.sample
            sampled  = _sample_windows(all_wins, _per_ds_test_held, rng)
            split["test"].extend(
                ((lc_id, case_id, target_N), w) for w in sampled
            )

        else:
            raise ValueError(f"LC {lc_id} is not assigned to any split rule.")

    return split

def print_split_summary(
    load_case_datasets: Dict[Tuple, MooringSequenceDatasetPositionTension],
    split_indices: Dict[str, List],
):
    """Print window counts per split, aggregated by (lc_id, target_N)."""
    print("----- SPLIT SUMMARY -----")
    for split_name, pairs in split_indices.items():
        # Aggregate by (lc_id, target_N) to keep output readable
        counts: Dict[Tuple, int] = {}
        for (lc_id, case_id, target_N), _ in pairs:
            key = (lc_id, target_N)
            counts[key] = counts.get(key, 0) + 1
        total = sum(counts.values())
        print(f"\n{split_name.upper()} total windows: {total}")
        for (lc_id, target_N) in sorted(counts):
            print(f"  LC {lc_id:02d} N={target_N:02d}: {counts[(lc_id, target_N)]} windows")


def _cap_split_windows(split_indices: Dict[str, List], cfg: TrainingConfig) -> Dict[str, List]:
    """Deterministically cap split sizes for small debug runs."""
    rng = random.Random(cfg.split_seed)
    capped = {}
    caps = {
        "train": cfg.max_train_windows,
        "val": cfg.max_val_windows,
        "test": cfg.max_test_windows,
    }
    for split_name, pairs in split_indices.items():
        pairs = list(pairs)
        cap = caps.get(split_name)
        if cap is not None and len(pairs) > cap:
            rng.shuffle(pairs)
            pairs = pairs[:cap]
            pairs.sort(key=lambda item: (item[0], item[1]))
        capped[split_name] = pairs
    return capped


In [10]:
# -------------------------------------------------------------
# subset dataset
# -------------------------------------------------------------

class MultiLoadCaseWindowSubset(Dataset):
    """
    Thin wrapper around multiple MooringSequenceDatasetPositionTension objects.
    Keys in load_case_datasets are (lc_id, case_id, target_N) 3-tuples.
    """
    def __init__(
        self,
        load_case_datasets: Dict[Tuple, MooringSequenceDatasetPositionTension],
        index_pairs: List,
    ):
        self.load_case_datasets = load_case_datasets
        self.index_pairs = index_pairs

    def __len__(self):
        return len(self.index_pairs)

    def __getitem__(self, idx):
        key, window_idx = self.index_pairs[idx]
        lc_id, case_id, node_count = key
        sample = self.load_case_datasets[key][window_idx]

        return {
            "graph":      sample["graph"],
            "x_seq":      sample["x_seq"],
            "edge_seq":   sample["edge_seq"],
            "y_seq":      sample["y_seq"],
            "start_idx":  sample["start_idx"],
            "lc_id":      lc_id,
            "case_id":    case_id,
            "node_count": node_count,
        }

def single_item_collate(batch):
    if len(batch) != 1:
        raise ValueError("single_item_collate expects batch_size=1.")
    return batch[0]


def bucketed_node_count_collate(batch):
    """
    Collate function for bucketed batching.
    All samples in the batch must share the same node_count.
    Returns stacked tensors [B, H, N, F] and graphs as List[Data].
    """
    nc = batch[0].get("node_count", batch[0]["x_seq"].shape[1])
    assert all(s.get("node_count", s["x_seq"].shape[1]) == nc for s in batch), (
        f"Batch contains mixed node counts: {[s.get('node_count') for s in batch]}"
    )
    assert all(s["x_seq"].shape    == batch[0]["x_seq"].shape    for s in batch), (
        f"x_seq shape mismatch within bucket batch: {[s['x_seq'].shape for s in batch]}"
    )
    assert all(s["edge_seq"].shape == batch[0]["edge_seq"].shape for s in batch), (
        f"edge_seq shape mismatch: {[s['edge_seq'].shape for s in batch]}"
    )
    assert all(s["y_seq"].shape    == batch[0]["y_seq"].shape    for s in batch), (
        f"y_seq shape mismatch: {[s['y_seq'].shape for s in batch]}"
    )
    return {
        # Keep each graph as a separate object; static features differ by lc_id/case_id
        # even when node_count matches
        "graphs":     [s["graph"]     for s in batch],              # List[Data], len=B
        "x_seq":      torch.stack([s["x_seq"]    for s in batch]),  # [B, H, N, F_node]
        "edge_seq":   torch.stack([s["edge_seq"] for s in batch]),  # [B, H, E, F_edge]
        "y_seq":      torch.stack([s["y_seq"]    for s in batch]),  # [B, P, N, 3]
        "start_idx":  [s["start_idx"]  for s in batch],
        "lc_id":      [s["lc_id"]      for s in batch],
        "case_id":    [s["case_id"]    for s in batch],
        "node_count": nc,
    }


In [11]:
# -------------------------------------------------------------
# normalization utilities
# -------------------------------------------------------------

class FeatureStandardizer:
    """
    Per-feature z-score standardization for selected feature columns.

    Works with tensors shaped:
    - node inputs:   [history_len, N, F]
    - edge inputs:   [history_len, E, F]
    - targets:       [future_len, N, F]
    """
    def __init__(self, feature_idx: Tuple[int, ...], eps: float = 1e-8):
        self.feature_idx = list(feature_idx)
        self.eps = eps
        self.mean = None
        self.std = None

    def fit_from_tensor_list(self, tensor_list: List[torch.Tensor]):
        """
        Aggregate across all dimensions except the last feature dimension.
        """
        selected = []
        for x in tensor_list:
            xs = x[..., self.feature_idx].reshape(-1, len(self.feature_idx))
            selected.append(xs)

        big = torch.cat(selected, dim=0)
        self.mean = big.mean(dim=0)
        self.std = big.std(dim=0, unbiased=False).clamp_min(self.eps)

    def transform(self, x: torch.Tensor) -> torch.Tensor:
        x = x.clone()
        x_sel = x[..., self.feature_idx]
        x[..., self.feature_idx] = (x_sel - self.mean.to(x.device)) / self.std.to(x.device)
        return x

    def inverse_transform(self, x: torch.Tensor) -> torch.Tensor:
        x = x.clone()
        x_sel = x[..., self.feature_idx]
        x[..., self.feature_idx] = x_sel * self.std.to(x.device) + self.mean.to(x.device)
        return x
    
def fit_normalizers_from_train_subset(
    train_subset: MultiLoadCaseWindowSubset,
    cfg: TrainingConfig,
):
    """
    Fit normalizers using vectorized batch-Welford online statistics.

    Draws cfg.n_fit_windows random windows from train_subset and updates
    running mean/variance one window at a time using pure tensor ops
    (no row-by-row Python loops, no accumulating window tensors in RAM).
    """
    node_norm   = FeatureStandardizer(cfg.node_continuous_idx,   eps=cfg.eps)
    edge_norm   = FeatureStandardizer(cfg.edge_continuous_idx,   eps=cfg.eps)
    target_norm = FeatureStandardizer(cfg.target_continuous_idx, eps=cfg.eps)

    rng     = random.Random(cfg.split_seed)
    indices = rng.sample(range(len(train_subset)), min(cfg.n_fit_windows, len(train_subset)))

    def _batch_welford_merge(count, mean, M2, raw_slice, feat_idx):
        """Merge one window into running statistics using vectorized tensor ops."""
        vals       = raw_slice[..., feat_idx].reshape(-1, len(feat_idx))   # [rows, F]
        n_new      = float(vals.shape[0])
        batch_mean = vals.mean(dim=0)
        batch_var  = vals.var(dim=0, unbiased=False)
        n_combined = count + n_new
        delta      = batch_mean - mean
        new_mean   = (count * mean + n_new * batch_mean) / n_combined
        new_M2     = M2 + batch_var * n_new + delta ** 2 * (count * n_new / n_combined)
        return n_combined, new_mean, new_M2

    def _fit_one(norm, key):
        n_feat          = len(norm.feature_idx)
        count           = 0.0
        mean            = torch.zeros(n_feat)
        M2              = torch.zeros(n_feat)
        for i in indices:
            raw         = train_subset[i][key]
            count, mean, M2 = _batch_welford_merge(count, mean, M2, raw, norm.feature_idx)
        norm.mean = mean
        norm.std  = (M2 / max(count, 1.0)).sqrt().clamp_min(norm.eps)

    _fit_one(node_norm,   "x_seq")
    _fit_one(edge_norm,   "edge_seq")
    _fit_one(target_norm, "y_seq")
    return node_norm, edge_norm, target_norm

class NormalizedSubset(Dataset):
    """
    Applies train-fitted normalization on the fly.
    """
    def __init__(
        self,
        base_subset: MultiLoadCaseWindowSubset,
        node_norm: FeatureStandardizer,
        edge_norm: FeatureStandardizer,
        target_norm: FeatureStandardizer,
    ):
        self.base_subset = base_subset
        self.node_norm = node_norm
        self.edge_norm = edge_norm
        self.target_norm = target_norm

    def __len__(self):
        return len(self.base_subset)

    def __getitem__(self, idx):
        sample = self.base_subset[idx]

        return {
            "graph":      sample["graph"],
            "x_seq":      self.node_norm.transform(sample["x_seq"]),
            "edge_seq":   self.edge_norm.transform(sample["edge_seq"]),
            "y_seq":      self.target_norm.transform(sample["y_seq"]),
            "start_idx":  sample["start_idx"],
            "lc_id":      sample["lc_id"],
            "case_id":    sample["case_id"],
            "node_count": sample["node_count"],
        }

In [12]:
# -------------------------------------------------------------
# dataloader builder
# -------------------------------------------------------------

class BucketedNodeCountBatchSampler(Sampler):
    """
    Buckets dataset indices by node_count; yields homogeneous batches.

    __init__: builds index->node_count map and full bucket lists (seeded, reproducible).
    __iter__: each epoch, sub-samples max_windows from the full pool (if set),
              shuffles within buckets, builds and shuffles batch list.
              Uses rng = random.Random(seed + epoch) so ordering is epoch-varying
              but reproducible.

    index_pairs: list of ((lc_id, case_id, node_count), window_idx)
    max_windows: per-epoch subsample cap (applied fresh each epoch in __iter__)
    """
    def __init__(self, index_pairs, batch_size, drop_last=False, seed=42, max_windows=None):
        from collections import defaultdict as _dd
        self.batch_size  = batch_size
        self.drop_last   = drop_last
        self.seed        = seed
        self.max_windows = max_windows
        self.epoch       = 0

        # Precompute flat index -> node_count for O(1) lookup in __iter__
        self._index_to_nc = {i: key[2] for i, (key, _) in enumerate(index_pairs)}

        # Build full bucket lists
        buckets = _dd(list)
        for i, nc in self._index_to_nc.items():
            buckets[nc].append(i)
        self._buckets = {nc: list(idxs) for nc, idxs in sorted(buckets.items())}

        # __len__ reflects the full-pool batch count.
        # train_one_epoch must not rely on len(loader) for the final grad-accum
        # flush when max_windows is active (iter may yield fewer batches).
        self._full_len = sum(
            len(idxs) // batch_size
            + (0 if drop_last else int(len(idxs) % batch_size > 0))
            for idxs in self._buckets.values()
        )

    def __iter__(self):
        import random as _random
        from collections import defaultdict as _dd

        rng = _random.Random(self.seed + self.epoch)
        self.epoch += 1

        # Sub-sample max_windows from full pool fresh each epoch
        all_indices = [i for idxs in self._buckets.values() for i in idxs]
        if self.max_windows is not None and len(all_indices) > self.max_windows:
            all_indices = rng.sample(all_indices, self.max_windows)

        # Re-bucket using precomputed O(1) lookup
        buckets = _dd(list)
        for i in all_indices:
            buckets[self._index_to_nc[i]].append(i)

        # Build batches within each bucket, then shuffle the batch list
        all_batches = []
        for nc in sorted(buckets):
            idxs = buckets[nc]
            rng.shuffle(idxs)
            for start in range(0, len(idxs), self.batch_size):
                b = idxs[start : start + self.batch_size]
                if self.drop_last and len(b) < self.batch_size:
                    continue
                all_batches.append(b)

        rng.shuffle(all_batches)
        yield from all_batches

    def __len__(self):
        return self._full_len


def build_dataloaders(
    load_case_datasets: Dict[int, MooringSequenceDatasetPositionTension],
    cfg: TrainingConfig,
):

    split_indices = build_leakage_aware_splits(load_case_datasets, cfg)
    split_indices = _cap_split_windows(split_indices, cfg)
    print_split_summary(load_case_datasets, split_indices)

    for split_name in ["train", "val", "test"]:
        if not split_indices[split_name]:
            raise ValueError(f"{split_name} split is empty. Adjust debug locs/cases or split settings.")

    train_raw = MultiLoadCaseWindowSubset(load_case_datasets, split_indices["train"])
    val_raw   = MultiLoadCaseWindowSubset(load_case_datasets, split_indices["val"])
    test_raw  = MultiLoadCaseWindowSubset(load_case_datasets, split_indices["test"])

    node_norm, edge_norm, target_norm = fit_normalizers_from_train_subset(train_raw, cfg)

    train_ds = NormalizedSubset(train_raw, node_norm, edge_norm, target_norm)
    val_ds   = NormalizedSubset(val_raw,   node_norm, edge_norm, target_norm)
    test_ds  = NormalizedSubset(test_raw,  node_norm, edge_norm, target_norm)

    _pf = 4 if cfg.num_workers > 0 else None   # prefetch_factor requires num_workers > 0

    if cfg.batch_size == 1:
        # ── single-sample path (unchanged) ───────────────────────────────
        # Per-epoch training sampler: replacement=True avoids allocating a full
        # permutation array, which would require ~1 GB for 270M windows.
        if cfg.max_train_windows_per_epoch is not None:
            n_samples     = min(cfg.max_train_windows_per_epoch, len(train_ds))
            train_sampler = RandomSampler(train_ds, replacement=True, num_samples=n_samples)
            train_shuffle = False
        else:
            train_sampler = None
            train_shuffle = True

        train_loader = DataLoader(
            train_ds,
            batch_size=1,
            sampler=train_sampler,
            shuffle=train_shuffle,
            collate_fn=single_item_collate,
            num_workers=cfg.num_workers,
            timeout=(600 if cfg.num_workers > 0 else 0),
            pin_memory=cfg.pin_memory,
            prefetch_factor=_pf,
            persistent_workers=False,
        )

    elif cfg.enable_bucketed_node_count_batching:
        # ── bucketed batching: one node_count per mini-batch ─────────────
        # max_windows is applied fresh each epoch inside the sampler's __iter__
        bucket_sampler = BucketedNodeCountBatchSampler(
            index_pairs=split_indices["train"],
            batch_size=cfg.batch_size,
            drop_last=cfg.drop_last_batch,
            seed=cfg.split_seed,
            max_windows=cfg.max_train_windows_per_epoch,
        )
        train_loader = DataLoader(
            train_ds,
            batch_sampler=bucket_sampler,
            collate_fn=bucketed_node_count_collate,
            num_workers=cfg.num_workers,
            timeout=(600 if cfg.num_workers > 0 else 0),
            pin_memory=cfg.pin_memory,
            prefetch_factor=_pf,
            persistent_workers=False,
        )

    else:
        raise ValueError(
            f"batch_size={cfg.batch_size} > 1 requires"
            " enable_bucketed_node_count_batching=True"
        )

    # Val / test: bucketed batching when batch_size > 1, else single-item.
    if cfg.batch_size > 1 and cfg.enable_bucketed_node_count_batching:
        val_loader = DataLoader(
            val_ds,
            batch_sampler=BucketedNodeCountBatchSampler(
                split_indices["val"], cfg.batch_size,
                drop_last=False, seed=cfg.split_seed, max_windows=None,
            ),
            collate_fn=bucketed_node_count_collate,
            num_workers=cfg.num_workers,
            timeout=(600 if cfg.num_workers > 0 else 0),
            pin_memory=cfg.pin_memory,
            prefetch_factor=_pf,
            persistent_workers=False,
        )
        test_loader = DataLoader(
            test_ds,
            batch_sampler=BucketedNodeCountBatchSampler(
                split_indices["test"], cfg.batch_size,
                drop_last=False, seed=cfg.split_seed, max_windows=None,
            ),
            collate_fn=bucketed_node_count_collate,
            num_workers=cfg.num_workers,
            timeout=(600 if cfg.num_workers > 0 else 0),
            pin_memory=cfg.pin_memory,
            prefetch_factor=_pf,
            persistent_workers=False,
        )
    else:
        val_loader = DataLoader(
            val_ds,
            batch_size=1,
            shuffle=False,
            collate_fn=single_item_collate,
            num_workers=cfg.num_workers,
            timeout=(600 if cfg.num_workers > 0 else 0),
            pin_memory=cfg.pin_memory,
            prefetch_factor=_pf,
            persistent_workers=False,
        )
        test_loader = DataLoader(
            test_ds,
            batch_size=1,
            shuffle=False,
            collate_fn=single_item_collate,
            num_workers=cfg.num_workers,
            timeout=(600 if cfg.num_workers > 0 else 0),
            pin_memory=cfg.pin_memory,
            prefetch_factor=_pf,
            persistent_workers=False,
        )

    norms = {
        "node_norm":   node_norm,
        "edge_norm":   edge_norm,
        "target_norm": target_norm,
    }

    return train_loader, val_loader, test_loader, norms, split_indices

In [13]:
# -------------------------------------------------------------
# loss and metrics
# -------------------------------------------------------------

class SequenceSmoothL1Loss(nn.Module):
    def __init__(self, beta: float = 1.0):
        super().__init__()
        self.loss_fn = nn.SmoothL1Loss(beta=beta)

    def forward(self, y_hat, y_true):
        return self.loss_fn(y_hat, y_true)
    
@torch.no_grad()
def compute_physical_metrics_per_target(
    y_hat_norm: torch.Tensor,
    y_true_norm: torch.Tensor,
    target_norm: FeatureStandardizer,
    target_names: List[str],
    mape_eps: float = 1e-8,
    mape_min: float = 1000.0,
    peak_mape_min: float = 1000.0,
):
    """
    Inputs:
        y_hat_norm, y_true_norm: [future_len, N, output_dim] in normalised space.
        mape_min         : per-step MAPE threshold on |true_tension| [N].
        peak_mape_min    : peak MAPE threshold on |true_peak_tension| [N].

    Returns a flat dict. Caller-important keys:
        MAE / RMSE / R2              : {target_name: float}  (mean over time+nodes)
        ss_res / y_sum / y_sq / n_pts: {target_name: float}  (raw sums for global R2)
        MAPE                         : {tension: float|nan, tension_valid_frac: float}
        peak_tension_MAE/_RMSE/_bias/_underpred_rate/_MAPE : float
        peak_tension_abs_errors      : List[float]  (per-node, for global p95)
        h{H:02d}_MAE                 : float  (mean over all targets+nodes, compat)
        h{H:02d}_MAE_{target_name}   : float  (per-target horizon MAE)
        temporal_diff_MAE_{name}     : float
    """
    import math as _math
    y_hat_phys  = target_norm.inverse_transform(y_hat_norm.detach().cpu().float())
    y_true_phys = target_norm.inverse_transform(y_true_norm.detach().cpu().float())
    error = y_hat_phys - y_true_phys   # [P, N, D]

    # Per-target MAE / RMSE (mean over time+nodes)
    mae  = error.abs().mean(dim=(0, 1))
    rmse = torch.sqrt((error ** 2).mean(dim=(0, 1)))

    # Per-window R2 (kept for backward compat; averaged by caller)
    y_true_mean_w = y_true_phys.mean(dim=(0, 1), keepdim=True)
    ss_res_w = (error ** 2).sum(dim=(0, 1))
    ss_tot_w = ((y_true_phys - y_true_mean_w) ** 2).sum(dim=(0, 1))
    r2_w     = 1.0 - ss_res_w / ss_tot_w.clamp_min(mape_eps)

    # Raw sums for global R2 accumulation in evaluate()
    ss_res_raw = (error ** 2).sum(dim=(0, 1))
    y_true_sum = y_true_phys.sum(dim=(0, 1))
    y_true_sq  = (y_true_phys ** 2).sum(dim=(0, 1))
    n_per_feat = float(y_true_phys.shape[0] * y_true_phys.shape[1])

    metrics = {
        "MAE":  {name: float(mae[i].item())   for i, name in enumerate(target_names)},
        "RMSE": {name: float(rmse[i].item())  for i, name in enumerate(target_names)},
        "R2":   {name: float(r2_w[i].item())  for i, name in enumerate(target_names)},
        "ss_res": {name: float(ss_res_raw[i].item()) for i, name in enumerate(target_names)},
        "y_sum":  {name: float(y_true_sum[i].item()) for i, name in enumerate(target_names)},
        "y_sq":   {name: float(y_true_sq[i].item())  for i, name in enumerate(target_names)},
        "n_pts":  n_per_feat,
        "MAPE": {},
    }

    # --- MAPE + peak-tension (tension only) ---
    if "tension" in target_names:
        t_idx  = target_names.index("tension")
        true_t = y_true_phys[..., t_idx]   # [P, N]
        pred_t = y_hat_phys[..., t_idx]

        mask = true_t.abs() > mape_min
        if mask.any():
            mape_val   = float(((pred_t - true_t).abs() / true_t.abs())[mask].mean().item() * 100.0)
            valid_frac = float(mask.float().mean().item())
        else:
            mape_val, valid_frac = float("nan"), 0.0
        metrics["MAPE"]["tension"]            = mape_val
        metrics["MAPE"]["tension_valid_frac"] = valid_frac

        true_peak = true_t.amax(dim=0)   # [N]
        pred_peak = pred_t.amax(dim=0)
        peak_err  = pred_peak - true_peak
        peak_abs  = peak_err.abs()
        metrics["peak_tension_MAE"]            = float(peak_abs.mean().item())
        metrics["peak_tension_RMSE"]           = float((peak_err ** 2).mean().sqrt().item())
        metrics["peak_tension_bias"]           = float(peak_err.mean().item())
        metrics["peak_tension_underpred_rate"] = float((peak_err < 0).float().mean().item() * 100.0)
        metrics["peak_tension_abs_errors"]     = peak_abs.tolist()

        pmask = true_peak.abs() > peak_mape_min
        if pmask.any():
            metrics["peak_tension_MAPE"] = float(
                (peak_abs[pmask] / true_peak.abs()[pmask]).mean().item() * 100.0
            )
        else:
            metrics["peak_tension_MAPE"] = float("nan")
    else:
        metrics["peak_tension_abs_errors"] = []

    # --- Per-target horizon MAE + backward-compat aggregated ---
    for h in (1, 10, 25, 50):
        idx = h - 1
        if idx < error.shape[0]:
            h_err = error[idx]   # [N, D]
            metrics[f"h{h:02d}_MAE"] = float(h_err.abs().mean().item())
            for i, name in enumerate(target_names):
                metrics[f"h{h:02d}_MAE_{name}"] = float(h_err[:, i].abs().mean().item())
        else:
            metrics[f"h{h:02d}_MAE"] = float("nan")
            for name in target_names:
                metrics[f"h{h:02d}_MAE_{name}"] = float("nan")

    # --- Temporal smoothness ---
    if error.shape[0] > 1:
        pred_diff  = y_hat_phys[1:] - y_hat_phys[:-1]
        true_diff  = y_true_phys[1:] - y_true_phys[:-1]
        diff_error = (pred_diff - true_diff).abs()
        for i, name in enumerate(target_names):
            metrics[f"temporal_diff_MAE_{name}"] = float(diff_error[..., i].mean().item())
    else:
        for name in target_names:
            metrics[f"temporal_diff_MAE_{name}"] = float("nan")

    return metrics


def compute_physical_loss(
    y_hat_norm: torch.Tensor,
    y_true_norm: torch.Tensor,
    target_norm,
    cfg,
    device: torch.device,
    log_sigma: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """
    Optional auxiliary loss in physical units, dimensionless via characteristic scales.
    All terms are zero when their config weights are zero.
    Inputs must be float32 (caller casts .float() before passing).
    Supports [P, N, D] or [B, P, N, D].
    """
    import torch.nn.functional as _F
    std_d  = target_norm.std.to(device).float()
    mean_d = target_norm.mean.to(device).float()

    def _sc(override, default):
        if override is not None:
            return torch.tensor(float(override), device=device)
        return default
    scales = torch.stack([
        _sc(cfg.phys_x_scale,       std_d[0]),
        _sc(cfg.phys_z_scale,       std_d[1]),
        _sc(cfg.phys_tension_scale, std_d[2]),
    ]).clamp_min(1e-6)

    y_hat_phys  = y_hat_norm.float()  * std_d + mean_d
    y_true_phys = y_true_norm.float() * std_d + mean_d
    total = torch.zeros(1, device=device)

    # --- Stage C2: Kendall multi-task uncertainty weighting -----------------
    # Replaces the 4 fixed w_* scalars with learnable log_sigma (= log sigma^2).
    # C2c: all 4 physical terms are Kendall-weighted (learnable log_sigma) as
    # 0.5*exp(-s_i)*L_i + 0.5*s_i. The underprediction term is a tau-quantile
    # (pinball) loss -- two-sided with an irreducible noise floor, so its sigma
    # converges (unlike the C2a one-sided relu hinge). Data loss keeps unit weight.
    if getattr(cfg, "use_kendall_weighting", False) and log_sigma is not None:
        import torch.nn.functional as _Fk
        L_mae = ((y_hat_phys - y_true_phys).abs() / scales).mean()        # term 0
        pred_t = y_hat_phys[..., 2]
        true_t = y_true_phys[..., 2]
        if pred_t.dim() == 2:
            pred_peak = pred_t.amax(dim=0); true_peak = true_t.amax(dim=0)
        else:
            pred_peak = pred_t.amax(dim=1); true_peak = true_t.amax(dim=1)
        T_sc = scales[2]
        L_peak  = (pred_peak - true_peak).abs().mean() / T_sc             # term 1
        _tau = float(getattr(cfg, "peak_pinball_tau", 0.9))              # C2c pinball quantile
        _d_under = (true_peak - pred_peak) / T_sc                        # >0 = underprediction
        L_under = torch.maximum(_tau * _d_under, (_tau - 1.0) * _d_under).mean()  # term 2 (pinball)
        if y_hat_phys.dim() == 4 and y_hat_phys.shape[1] > 1:
            pred_diff = y_hat_phys[:, 1:] - y_hat_phys[:, :-1]
            true_diff = y_true_phys[:, 1:] - y_true_phys[:, :-1]
        elif y_hat_phys.dim() == 3 and y_hat_phys.shape[0] > 1:
            pred_diff = y_hat_phys[1:] - y_hat_phys[:-1]
            true_diff = y_true_phys[1:] - y_true_phys[:-1]
        else:
            pred_diff = true_diff = None
        if pred_diff is not None:
            L_temp = ((pred_diff - true_diff).abs() / scales).mean()      # term 3
        else:
            L_temp = torch.zeros((), device=device)
        s = log_sigma.to(device)                                         # s_i = log sigma_i^2
        # C2c: all 4 terms Kendall-weighted; the underprediction term is the two-sided
        # pinball L_under above (noise floor present), so its sigma converges too.
        terms = torch.stack([L_mae, L_peak, L_under, L_temp])
        kendall = (0.5 * torch.exp(-s) * terms + 0.5 * s).sum()
        return kendall

    if cfg.w_phys_mae > 0.0:
        smae  = ((y_hat_phys - y_true_phys).abs() / scales).mean()
        total = total + cfg.w_phys_mae * smae

    if cfg.w_peak_tension > 0.0 or cfg.w_peak_underprediction > 0.0:
        pred_t = y_hat_phys[..., 2]
        true_t = y_true_phys[..., 2]
        if pred_t.dim() == 2:
            pred_peak = pred_t.amax(dim=0)
            true_peak = true_t.amax(dim=0)
        elif pred_t.dim() == 3:
            pred_peak = pred_t.amax(dim=1)
            true_peak = true_t.amax(dim=1)
        else:
            raise ValueError(f"Unexpected tension shape after slicing: {pred_t.shape}")
        T_sc = scales[2]
        if cfg.w_peak_tension > 0.0:
            total = total + cfg.w_peak_tension * (pred_peak - true_peak).abs().mean() / T_sc
        if cfg.w_peak_underprediction > 0.0:
            total = total + cfg.w_peak_underprediction * _F.relu(true_peak - pred_peak).mean() / T_sc

    if cfg.w_temporal_smoothness > 0.0:
        if y_hat_phys.dim() == 4 and y_hat_phys.shape[1] > 1:
            pred_diff = y_hat_phys[:, 1:] - y_hat_phys[:, :-1]
            true_diff = y_true_phys[:, 1:] - y_true_phys[:, :-1]
        elif y_hat_phys.dim() == 3 and y_hat_phys.shape[0] > 1:
            pred_diff = y_hat_phys[1:] - y_hat_phys[:-1]
            true_diff = y_true_phys[1:] - y_true_phys[:-1]
        else:
            pred_diff = true_diff = None
        if pred_diff is not None:
            total = total + cfg.w_temporal_smoothness * (
                (pred_diff - true_diff).abs() / scales
            ).mean()

    return total.squeeze()


In [14]:
# -------------------------------------------------------------
# model/optimizer setup
# -------------------------------------------------------------

def build_model_from_config(
    example_dataset: MooringSequenceDatasetPositionTension,
    cfg: TrainingConfig,
    device: torch.device,
):
    """Build MooringGATLSTM using architecture dims from cfg."""
    model = MooringGATLSTM(
        n_static_node_features=example_dataset.graph_data.x.shape[1],
        n_dynamic_node_features=example_dataset.n_dynamic_node_features,
        n_static_edge_features=example_dataset.graph_data.edge_attr.shape[1],
        n_dynamic_edge_features=example_dataset.n_dynamic_edge_features,
        future_len=example_dataset.future_len,
        output_dim=example_dataset.n_targets,

        # architecture dims from TrainingConfig
        gat_hidden_dim=cfg.gat_hidden_dim,
        gat_out_dim=cfg.gat_out_dim,
        num_heads=cfg.num_heads,
        gat_dropout=cfg.gat_dropout,
        gat_use_layernorm=True,
        add_residual_projection=True,

        lstm_hidden_dim=cfg.lstm_hidden_dim,
        num_lstm_layers=cfg.num_lstm_layers,
        lstm_dropout=cfg.lstm_dropout,
        bidirectional=False,
        lstm_use_layernorm=True,
        use_last_timestep=True,

        head_hidden_dim=cfg.head_hidden_dim,
        head_dropout=cfg.head_dropout,
    ).to(device)
    return model


def build_model_from_dataset_example(
    example_dataset: MooringSequenceDatasetPositionTension,
    device: torch.device,
    cfg: TrainingConfig = None,
):
    """Backwards-compatible wrapper. Pass cfg or default TrainingConfig() is used."""
    return build_model_from_config(example_dataset, cfg if cfg is not None else TrainingConfig(), device)

def build_optimizer_and_scheduler(
    model: nn.Module,
    cfg: TrainingConfig,
):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    scheduler = None
    if cfg.use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=cfg.scheduler_factor,
            patience=cfg.scheduler_patience,
        )

    return optimizer, scheduler

In [15]:
# -------------------------------------------------------------
# training and validation loops
# -------------------------------------------------------------


def move_sample_to_device(sample, device):
    result = {
        "x_seq":      sample["x_seq"].to(device, non_blocking=True),
        "edge_seq":   sample["edge_seq"].to(device, non_blocking=True),
        "y_seq":      sample["y_seq"].to(device, non_blocking=True),
        "start_idx":  sample["start_idx"],
        "lc_id":      sample.get("lc_id"),
        "case_id":    sample.get("case_id"),
        "node_count": sample.get("node_count"),
    }
    if "graphs" in sample:
        # Batched: clone each graph individually before moving to device
        # (required to avoid mutating dataset-internal graph objects)
        result["graphs"] = [g.clone().to(device) for g in sample["graphs"]]
    else:
        result["graph"] = sample["graph"].clone().to(device)
    return result

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    cfg: TrainingConfig,
    scaler,
    target_norm=None,
    target_names=None,
):
    """
    Returns (avg_total_loss, avg_grad_norm, train_phys_metrics, avg_data_loss, avg_phys_loss).
    train_phys_metrics is {} when cfg.compute_train_physical_metrics is False.
    avg_data_loss is the unscaled, pre-division normalized SmoothL1 loss.
    """
    import math as _math
    model.train()

    amp_enabled       = cfg.use_amp and device.type == "cuda"
    running_data_loss = 0.0
    running_phys_loss = 0.0
    running_grad_norm = 0.0
    num_batches       = 0
    optimizer_steps   = 0
    optimizer.zero_grad(set_to_none=True)
    pending_grad = False
    n_skipped = 0

    do_train_phys = (
        cfg.compute_train_physical_metrics
        and target_norm is not None
        and target_names is not None
    )
    _tnames = target_names or []
    phys_struct = {k: {n: 0.0 for n in _tnames}
                   for k in ("MAE", "RMSE", "R2", "ss_res", "y_sum", "y_sq")}
    phys_n_pts  = {n: 0.0 for n in _tnames}
    phys_flat   = {}
    phys_flat_c = {}
    phys_peak_abs = []
    phys_nb     = 0

    for step_idx, sample in enumerate(loader):
        sample = move_sample_to_device(sample, device)

        if cfg.debug_batch_shapes and step_idx < 3:
            nc = sample.get("node_count", "?")
            gs = len(sample["graphs"]) if "graphs" in sample else 1
            print(f"  [batch debug] step={step_idx}  x_seq={tuple(sample['x_seq'].shape)}"
                  f"  node_count={nc}  n_graphs={gs}")

        with torch.amp.autocast("cuda", enabled=amp_enabled):
            graph_or_graphs = sample.get("graphs", sample.get("graph"))
            y_hat     = model(graph_or_graphs, sample["x_seq"], sample["edge_seq"])
            data_loss = criterion(y_hat, sample["y_seq"])

        phys_loss_val = 0.0
        if cfg.use_physical_loss_terms and target_norm is not None:
            with torch.amp.autocast("cuda", enabled=False):
                phys_loss = compute_physical_loss(
                    y_hat.float(), sample["y_seq"].float(),
                    target_norm, cfg, device,
                    log_sigma=getattr(model, "log_sigma", None),
                )
            phys_loss_val = phys_loss.item()
            loss = data_loss + phys_loss
        else:
            loss = data_loss

        if cfg.accumulation_steps > 1:
            loss = loss / cfg.accumulation_steps

        # per-batch NaN/Inf loss guard: skip a corrupt batch instead of crashing the run
        if not torch.isfinite(loss):
            n_skipped += 1
            if n_skipped <= 10:
                print(f"  [nan-guard] non-finite loss at step {step_idx}; skipping batch (corrupt data?)")
            continue
        scaler.scale(loss).backward()
        pending_grad = True

        if (step_idx + 1) % cfg.accumulation_steps == 0:
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                cfg.grad_clip_max_norm if cfg.grad_clip_max_norm is not None else float("inf"),
            )
            running_grad_norm += float(grad_norm)
            optimizer_steps   += 1
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            pending_grad = False

        # Log the unscaled (pre-division) data_loss for interpretable CSV
        running_data_loss += data_loss.item()
        running_phys_loss += phys_loss_val
        num_batches        += 1

        if do_train_phys and phys_nb < cfg.max_train_metric_batches:
            y_hat_det = y_hat.detach()
            y_seq     = sample["y_seq"]
            items_h = [y_hat_det[b] for b in range(y_hat_det.shape[0])] if y_hat_det.dim() == 4 else [y_hat_det]
            items_t = [y_seq[b]     for b in range(y_hat_det.shape[0])] if y_hat_det.dim() == 4 else [y_seq]
            for yh, yt in zip(items_h, items_t):
                m = compute_physical_metrics_per_target(
                    y_hat_norm=yh, y_true_norm=yt,
                    target_norm=target_norm, target_names=target_names,
                )
                phys_nb += 1
                for metric in ("MAE", "RMSE", "R2", "ss_res", "y_sum", "y_sq"):
                    for n in target_names:
                        phys_struct[metric][n] += m[metric][n]
                for n in target_names:
                    phys_n_pts[n] += m["n_pts"]
                phys_peak_abs.extend(m.get("peak_tension_abs_errors", []))
                _skip = {"MAE","RMSE","R2","MAPE","ss_res","y_sum","y_sq",
                         "n_pts","peak_tension_abs_errors"}
                for k in list(m["MAPE"].keys()) + [k for k in m if k not in _skip]:
                    val = m["MAPE"].get(k) if k in m["MAPE"] else m[k]
                    import math as _m2
                    if k not in phys_flat:
                        phys_flat[k]   = 0.0
                        phys_flat_c[k] = 0
                    if not _m2.isnan(val):
                        phys_flat[k]   += val
                        phys_flat_c[k] += 1

    if pending_grad:
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            cfg.grad_clip_max_norm if cfg.grad_clip_max_norm is not None else float("inf"),
        )
        running_grad_norm += float(grad_norm)
        optimizer_steps   += 1
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    avg_data_loss  = running_data_loss / max(1, num_batches)
    avg_phys_loss  = running_phys_loss / max(1, num_batches)
    avg_total_loss = avg_data_loss + avg_phys_loss
    avg_grad_norm  = running_grad_norm / max(1, optimizer_steps)

    train_phys_metrics = {}
    if do_train_phys and phys_nb > 0:
        _meps = 1e-8
        def _gr2(ss, ys, yq, np_, tnames):
            out = {}
            for n in tnames:
                n_t = np_[n]
                if n_t < 2:
                    out[n] = float("nan"); continue
                gm = ys[n] / n_t
                ss_tot = yq[n] - n_t * gm ** 2
                out[n] = float(1.0 - ss[n] / max(ss_tot, _meps))
            return out
        train_phys_metrics = {
            "MAE":       {n: phys_struct["MAE"][n]  / phys_nb for n in target_names},
            "RMSE":      {n: phys_struct["RMSE"][n] / phys_nb for n in target_names},
            "R2":        {n: phys_struct["R2"][n]   / phys_nb for n in target_names},
            "global_R2": _gr2(phys_struct["ss_res"], phys_struct["y_sum"],
                               phys_struct["y_sq"], phys_n_pts, target_names),
            "MAPE": {},
        }
        for k, s in phys_flat.items():
            cnt = phys_flat_c[k]
            val = s / cnt if cnt > 0 else float("nan")
            if k in ("tension", "tension_valid_frac"):
                train_phys_metrics["MAPE"][k] = val
            else:
                train_phys_metrics[k] = val
        if phys_peak_abs:
            sp = sorted(phys_peak_abs)
            p95_idx = max(0, int(0.95 * len(sp)) - 1)
            train_phys_metrics["peak_tension_p95_abs_error"] = float(sp[p95_idx])
            train_phys_metrics["peak_tension_max_abs_error"] = float(sp[-1])

    return avg_total_loss, avg_grad_norm, train_phys_metrics, avg_data_loss, avg_phys_loss


@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
    target_norm: FeatureStandardizer,
    target_names: List[str],
    mape_min: float = 1000.0,
    per_nc_metrics: Optional[dict] = None,
    cfg=None,
):
    """
    Evaluate model over loader. Returns (avg_loss, avg_metrics).

    When cfg.use_physical_loss_terms is True, val_loss includes the physical
    loss terms (same objective as train_total_loss). When cfg=None or
    use_physical_loss_terms=False, val_loss equals the normalized SmoothL1 only.

    avg_metrics adds: global_R2, per-target horizon MAE, temporal_diff_MAE,
    true aggregated peak_tension_p95_abs_error/max_abs_error, peak_tension_MAPE,
    val_data_loss_norm, val_phys_loss.

    per_nc_metrics (optional out-param): if a dict is passed, it is populated
    as {node_count: avg_metrics_dict} with per-node-count breakdown.
    """
    import math as _math
    from collections import defaultdict as _dd
    model.eval()

    running_data_loss = 0.0
    running_phys_loss = 0.0
    num_batches       = 0

    struct_sums = {
        "MAE":    {n: 0.0 for n in target_names},
        "RMSE":   {n: 0.0 for n in target_names},
        "R2":     {n: 0.0 for n in target_names},
        "ss_res": {n: 0.0 for n in target_names},
        "y_sum":  {n: 0.0 for n in target_names},
        "y_sq":   {n: 0.0 for n in target_names},
        "n_pts":  {n: 0.0 for n in target_names},
    }
    flat_sums   = {}
    flat_counts = {}
    all_peak_abs_errors = []

    do_nc = per_nc_metrics is not None
    nc_struct   = _dd(lambda: {k: {n: 0.0 for n in target_names}
                                for k in ("MAE","RMSE","R2","ss_res","y_sum","y_sq","n_pts")})
    nc_flat     = _dd(dict)
    nc_flat_c   = _dd(dict)
    nc_peak_abs = _dd(list)
    nc_nb       = _dd(int)

    for sample in loader:
        sample = move_sample_to_device(sample, device)
        nc_val = sample.get("node_count")

        graph_or_graphs = sample.get("graphs", sample.get("graph"))
        y_hat_out = model(graph_or_graphs, sample["x_seq"], sample["edge_seq"])

        if y_hat_out.dim() == 4:
            y_hat_items  = [y_hat_out[b]        for b in range(y_hat_out.shape[0])]
            y_true_items = [sample["y_seq"][b]  for b in range(y_hat_out.shape[0])]
        else:
            y_hat_items  = [y_hat_out]
            y_true_items = [sample["y_seq"]]

        for y_hat, y_true in zip(y_hat_items, y_true_items):
            loss = criterion(y_hat, y_true)
            running_data_loss += loss.item()
            _phys_val = 0.0
            if cfg is not None and cfg.use_physical_loss_terms:
                with torch.amp.autocast("cuda", enabled=False):
                    _pl = compute_physical_loss(
                        y_hat.float(), y_true.float(), target_norm, cfg, device,
                        log_sigma=getattr(model, "log_sigma", None),
                    )
                _phys_val = _pl.item()
            running_phys_loss += _phys_val
            num_batches += 1

            m = compute_physical_metrics_per_target(
                y_hat_norm=y_hat,
                y_true_norm=y_true,
                target_norm=target_norm,
                target_names=target_names,
                mape_min=mape_min,
            )

            for metric in ("MAE", "RMSE", "R2"):
                for n in target_names:
                    struct_sums[metric][n] += m[metric][n]
            for rk in ("ss_res", "y_sum", "y_sq"):
                for n in target_names:
                    struct_sums[rk][n] += m[rk][n]
            for n in target_names:
                struct_sums["n_pts"][n] += m["n_pts"]
            all_peak_abs_errors.extend(m.get("peak_tension_abs_errors", []))

            _skip = {"MAE","RMSE","R2","MAPE","ss_res","y_sum","y_sq",
                     "n_pts","peak_tension_abs_errors"}
            flat_keys = list(m["MAPE"].keys()) + [k for k in m if k not in _skip]
            for k in flat_keys:
                val = m["MAPE"].get(k) if k in m["MAPE"] else m[k]
                if k not in flat_sums:
                    flat_sums[k]   = 0.0
                    flat_counts[k] = 0
                if not _math.isnan(val):
                    flat_sums[k]   += val
                    flat_counts[k] += 1

            if do_nc and nc_val is not None:
                nc_nb[nc_val] += 1
                for metric in ("MAE", "RMSE", "R2"):
                    for n in target_names:
                        nc_struct[nc_val][metric][n] += m[metric][n]
                for rk in ("ss_res", "y_sum", "y_sq"):
                    for n in target_names:
                        nc_struct[nc_val][rk][n] += m[rk][n]
                for n in target_names:
                    nc_struct[nc_val]["n_pts"][n] += m["n_pts"]
                nc_peak_abs[nc_val].extend(m.get("peak_tension_abs_errors", []))
                for k in flat_keys:
                    val = m["MAPE"].get(k) if k in m["MAPE"] else m[k]
                    if k not in nc_flat[nc_val]:
                        nc_flat[nc_val][k]   = 0.0
                        nc_flat_c[nc_val][k] = 0
                    if not _math.isnan(val):
                        nc_flat[nc_val][k]   += val
                        nc_flat_c[nc_val][k] += 1

    avg_data_loss = running_data_loss / max(1, num_batches)
    avg_phys_loss = running_phys_loss / max(1, num_batches)
    avg_loss      = avg_data_loss + avg_phys_loss
    _meps = 1e-8

    def _global_r2(ss_d, ys_d, yq_d, np_d, tnames):
        out = {}
        for n in tnames:
            n_t = np_d[n]
            if n_t < 2:
                out[n] = float("nan"); continue
            gm = ys_d[n] / n_t
            ss_tot = yq_d[n] - n_t * gm ** 2
            out[n] = float(1.0 - ss_d[n] / max(ss_tot, _meps))
        return out

    def _agg_p95_max(errs):
        if not errs:
            return float("nan"), float("nan")
        se = sorted(errs)
        return float(se[max(0, int(0.95 * len(se)) - 1)]), float(se[-1])

    def _build_avg(struct_s, flat_s, flat_c, nb, pk_list):
        avg = {
            "MAE":       {n: struct_s["MAE"][n]  / max(1, nb) for n in target_names},
            "RMSE":      {n: struct_s["RMSE"][n] / max(1, nb) for n in target_names},
            "R2":        {n: struct_s["R2"][n]   / max(1, nb) for n in target_names},
            "global_R2": _global_r2(
                struct_s["ss_res"], struct_s["y_sum"], struct_s["y_sq"],
                struct_s["n_pts"], target_names
            ),
            "MAPE": {},
        }
        for k, s in flat_s.items():
            cnt = flat_c[k]
            val = s / cnt if cnt > 0 else float("nan")
            if k in ("tension", "tension_valid_frac"):
                avg["MAPE"][k] = val
            else:
                avg[k] = val
        p95, pmax = _agg_p95_max(pk_list)
        avg["peak_tension_p95_abs_error"] = p95
        avg["peak_tension_max_abs_error"] = pmax
        return avg

    avg_metrics = _build_avg(
        struct_sums, flat_sums, flat_counts, num_batches, all_peak_abs_errors
    )

    if do_nc:
        per_nc_metrics.clear()
        for nc_key, nb_val in nc_nb.items():
            per_nc_metrics[nc_key] = _build_avg(
                nc_struct[nc_key], nc_flat[nc_key], nc_flat_c[nc_key],
                nb_val, nc_peak_abs[nc_key]
            )

    avg_metrics["val_data_loss_norm"] = avg_data_loss
    avg_metrics["val_phys_loss"]      = avg_phys_loss
    return avg_loss, avg_metrics


In [16]:
# -------------------------------------------------------------
# full training runner with checkpointing and early stopping
# -------------------------------------------------------------

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterion,
    device,
    target_norm: FeatureStandardizer,
    target_names: List[str],
    cfg: TrainingConfig,
    run_seed: int,
):
    import csv as _csv, dataclasses as _dc, json as _json, math as _math

    checkpoint_dir = Path(cfg.checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_model_path = checkpoint_dir / f"best_mooring_gat_lstm_seed_{run_seed}.pt"

    # -- config.json: written once at training start ---------------------------
    config_path = checkpoint_dir / "config.json"
    with open(config_path, "w", encoding="utf-8") as _f:
        _json.dump(_dc.asdict(cfg), _f, indent=2, default=str)

    # -- metrics_history.csv: stable fieldnames determined before loop ---------
    _h_tags = ["h01", "h10", "h25", "h50"]
    _tgt    = target_names
    _csv_base_fields = [
        "epoch", "lr",
        "train_loss",            # alias for train_total_loss — backward compat
        "train_total_loss", "train_data_loss_norm", "train_phys_loss",
        "grad_norm", "amp_scale",
        "sigma_phys_mae", "sigma_peak_tension",
        "sigma_peak_underpred", "sigma_temporal",
    ]
    _train_phys_fields = (
        [f"train_MAE_{n}"       for n in _tgt] +
        [f"train_RMSE_{n}"      for n in _tgt] +
        [f"train_global_R2_{n}" for n in _tgt] +
        ["train_MAPE_tension", "train_MAPE_tension_valid_frac"] +
        [f"train_{h}_MAE_{n}" for h in _h_tags for n in _tgt] +
        ["train_peak_tension_MAE", "train_peak_tension_bias",
         "train_peak_tension_underpred_rate",
         "train_peak_tension_p95_abs_error"]
    )
    _csv_val_fields = (
        [f"val_MAE_{n}"       for n in _tgt] +
        [f"val_RMSE_{n}"      for n in _tgt] +
        [f"val_R2_{n}"        for n in _tgt] +
        [f"val_global_R2_{n}" for n in _tgt] +
        ["val_MAPE_tension", "val_MAPE_tension_valid_frac",
         "val_peak_tension_MAE", "val_peak_tension_RMSE",
         "val_peak_tension_bias", "val_peak_tension_underpred_rate",
         "val_peak_tension_MAPE",
         "val_peak_tension_p95_abs_error", "val_peak_tension_max_abs_error"] +
        ["val_h01_MAE", "val_h10_MAE", "val_h25_MAE", "val_h50_MAE"] +
        [f"val_{h}_MAE_{n}"          for h in _h_tags for n in _tgt] +
        [f"val_temporal_diff_MAE_{n}" for n in _tgt] +
        ["val_data_loss_norm", "val_phys_loss", "val_loss", "selection_score"]
    )
    _csv_fields  = _csv_base_fields + _train_phys_fields + _csv_val_fields
    history_path = checkpoint_dir / "metrics_history.csv"
    _csv_file    = open(history_path, "w", newline="", encoding="utf-8")
    _csv_writer  = _csv.DictWriter(_csv_file, fieldnames=_csv_fields, extrasaction="ignore")
    _csv_writer.writeheader()
    # -- metrics_by_node_count.csv -------------------------------------------
    _nc_csv_fields = [
        "epoch", "split", "node_count",
        "MAE_x_abs", "MAE_z_abs", "MAE_tension",
        "RMSE_x_abs", "RMSE_z_abs", "RMSE_tension",
        "R2_x_abs", "R2_z_abs", "R2_tension",
        "global_R2_x_abs", "global_R2_z_abs", "global_R2_tension",
        "MAPE_tension",
        "peak_tension_MAE", "peak_tension_bias",
        "peak_tension_underpred_rate", "peak_tension_MAPE",
        "peak_tension_p95_abs_error", "peak_tension_max_abs_error",
        "temporal_diff_MAE_x_abs", "temporal_diff_MAE_z_abs",
        "temporal_diff_MAE_tension",
    ]
    _nc_path   = checkpoint_dir / "metrics_by_node_count.csv"
    _nc_file   = open(_nc_path, "w", newline="", encoding="utf-8")
    _nc_writer = _csv.DictWriter(_nc_file, fieldnames=_nc_csv_fields, extrasaction="ignore")
    _nc_writer.writeheader()
    history = {
        "train_loss": [],
        "grad_norm":  [],
        "val_loss":   [],
        "val_metrics": [],
    }

    # AMP GradScaler — no-op on CPU (enabled=False passes through cleanly)
    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and device.type == "cuda"))
    best_state                 = None
    epochs_without_improvement = 0
    start_epoch                = 1
    best_selection_score       = float("inf")
    best_val_loss_at_selection = float("inf")

    # -- checkpoint resume -----------------------------------------------
    if cfg.resume_from_checkpoint is not None:
        _ckpt = torch.load(cfg.resume_from_checkpoint, map_location=device, weights_only=False)
        model.load_state_dict(_ckpt["model_state_dict"])
        optimizer.load_state_dict(_ckpt["optimizer_state_dict"])
        if scheduler is not None and _ckpt.get("scheduler_state_dict"):
            scheduler.load_state_dict(_ckpt["scheduler_state_dict"])
        if _ckpt.get("scaler_state_dict"):
            scaler.load_state_dict(_ckpt["scaler_state_dict"])
        start_epoch = _ckpt.get("epoch", 0) + 1
        best_selection_score       = _ckpt.get("best_selection_score",
                                                _ckpt.get("best_val_loss", float("inf")))
        best_val_loss_at_selection = _ckpt.get("best_val_loss_at_selection",
                                                _ckpt.get("best_val_loss", float("inf")))
        print(f"Resumed from {cfg.resume_from_checkpoint}, "
              f"starting at epoch {start_epoch}, "
              f"best_selection_score={best_selection_score:.6f}")

    for epoch in range(start_epoch, cfg.num_epochs + 1):
        train_total_loss, grad_norm, train_phys, train_data_loss, train_phys_loss = \
            train_one_epoch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                criterion=criterion,
                device=device,
                cfg=cfg,
                scaler=scaler,
                target_norm=target_norm,
                target_names=target_names,
            )

        history["train_loss"].append(train_total_loss)
        history["grad_norm"].append(grad_norm)

        amp_scale  = scaler.get_scale()
        current_lr = optimizer.param_groups[0]["lr"]

        if _math.isnan(train_total_loss) or _math.isinf(train_total_loss):
            print(f"CRITICAL: train_total_loss={train_total_loss} at epoch {epoch} -- "
                  f"AMP overflow or corrupt batch. Stopping training.")
            break
        if amp_scale < 1.0:
            print(f"WARNING: AMP loss scale={amp_scale:.4f} at epoch {epoch} -- "
                  f"gradients likely underflowing. Set use_amp=False if loss stalls.")

        _row = {
            "epoch":                epoch,
            "lr":                   current_lr,
            "train_loss":           train_total_loss,
            "train_total_loss":     train_total_loss,
            "train_data_loss_norm": train_data_loss,
            "train_phys_loss":      train_phys_loss,
            "grad_norm":            grad_norm,
            "amp_scale":            amp_scale,
        }
        if getattr(cfg, "use_kendall_weighting", False) and hasattr(model, "log_sigma"):
            _sig = torch.exp(0.5 * model.log_sigma.detach().cpu()).tolist()  # sigma_i = exp(s_i/2)
            _row["sigma_phys_mae"]       = _sig[0]
            _row["sigma_peak_tension"]   = _sig[1]
            _row["sigma_peak_underpred"] = _sig[2]
            _row["sigma_temporal"]       = _sig[3]
        if train_phys:
            _row.update({f"train_MAE_{n}":  train_phys["MAE"].get(n, "")  for n in target_names})
            _row.update({f"train_RMSE_{n}": train_phys["RMSE"].get(n, "") for n in target_names})
            _row.update({f"train_global_R2_{n}": train_phys["global_R2"].get(n, "") for n in target_names})
            _row["train_MAPE_tension"]              = train_phys["MAPE"].get("tension", "")
            _row["train_MAPE_tension_valid_frac"]   = train_phys["MAPE"].get("tension_valid_frac", "")
            _row["train_peak_tension_MAE"]           = train_phys.get("peak_tension_MAE", "")
            _row["train_peak_tension_bias"]          = train_phys.get("peak_tension_bias", "")
            _row["train_peak_tension_underpred_rate"]= train_phys.get("peak_tension_underpred_rate", "")
            _row["train_peak_tension_p95_abs_error"] = train_phys.get("peak_tension_p95_abs_error", "")
            for h in (1, 10, 25, 50):
                for n in target_names:
                    _row[f"train_h{h:02d}_MAE_{n}"] = train_phys.get(f"h{h:02d}_MAE_{n}", "")

        is_last_epoch = (epoch == cfg.num_epochs)
        # validate_every: run validation every N epochs and always on the last epoch.
        # Early-stopping patience counts validation events, not training epochs.
        do_validate = (epoch % cfg.validate_every == 0) or is_last_epoch

        if do_validate:
            _nc_out = {}
            val_loss, val_metrics = evaluate(
                model=model,
                loader=val_loader,
                criterion=criterion,
                device=device,
                target_norm=target_norm,
                target_names=target_names,
                mape_min=cfg.tension_mape_min,
                per_nc_metrics=_nc_out,
                cfg=cfg,
            )

            if scheduler is not None:
                scheduler.step(val_loss)

            history["val_loss"].append(val_loss)
            history["val_metrics"].append(val_metrics)

            mae_str  = " | ".join([f"val_MAE_{k}={v:.6f}"  for k, v in val_metrics["MAE"].items()])
            rmse_str = " | ".join([f"val_RMSE_{k}={v:.6f}" for k, v in val_metrics["RMSE"].items()])
            r2_str   = " | ".join([f"val_R2_{k}={v:.6f}"   for k, v in val_metrics["R2"].items()])
            extra_parts = [mae_str, rmse_str, r2_str]
            if "tension" in val_metrics["MAPE"]:
                extra_parts.append(f"val_MAPE_tension={val_metrics['MAPE']['tension']:.6f}")
            metrics_str = " | ".join(extra_parts)
            if getattr(cfg, "use_kendall_weighting", False) and hasattr(model, "log_sigma"):
                _sg = torch.exp(0.5 * model.log_sigma.detach().cpu()).tolist()
                metrics_str = metrics_str + (
                    f" | sigma[mae,peak,under,temp]="
                    f"[{_sg[0]:.3f},{_sg[1]:.3f},{_sg[2]:.3f},{_sg[3]:.3f}]"
                )

            print(
                f"Epoch {epoch:03d} | "
                f"train_loss={train_total_loss:.6f} (data={train_data_loss:.6f}) | "
                f"grad_norm={grad_norm:.4f} | amp_scale={amp_scale:.0f} | "
                f"val_loss={val_loss:.6f} | "
                f"{metrics_str}"
            )

            # -- append a completed row with val metrics to CSV ---------------
            _row.update({
                "val_loss":                        val_loss,
                "val_data_loss_norm":              val_metrics.get("val_data_loss_norm", ""),
                "val_phys_loss":                   val_metrics.get("val_phys_loss", ""),
                "val_MAPE_tension":                val_metrics["MAPE"].get("tension", ""),
                "val_MAPE_tension_valid_frac":     val_metrics["MAPE"].get("tension_valid_frac", ""),
                "val_peak_tension_MAE":            val_metrics.get("peak_tension_MAE", ""),
                "val_peak_tension_RMSE":           val_metrics.get("peak_tension_RMSE", ""),
                "val_peak_tension_bias":           val_metrics.get("peak_tension_bias", ""),
                "val_peak_tension_underpred_rate": val_metrics.get("peak_tension_underpred_rate", ""),
                "val_peak_tension_MAPE":           val_metrics.get("peak_tension_MAPE", ""),
                "val_peak_tension_p95_abs_error":  val_metrics.get("peak_tension_p95_abs_error", ""),
                "val_peak_tension_max_abs_error":  val_metrics.get("peak_tension_max_abs_error", ""),
                "val_h01_MAE": val_metrics.get("h01_MAE", ""),
                "val_h10_MAE": val_metrics.get("h10_MAE", ""),
                "val_h25_MAE": val_metrics.get("h25_MAE", ""),
                "val_h50_MAE": val_metrics.get("h50_MAE", ""),
                **{f"val_MAE_{n}":       val_metrics["MAE"][n]       for n in target_names},
                **{f"val_RMSE_{n}":      val_metrics["RMSE"][n]      for n in target_names},
                **{f"val_R2_{n}":        val_metrics["R2"][n]        for n in target_names},
                **{f"val_global_R2_{n}": val_metrics["global_R2"][n] for n in target_names},
            })
            for h in (1, 10, 25, 50):
                for n in target_names:
                    _row[f"val_h{h:02d}_MAE_{n}"] = val_metrics.get(f"h{h:02d}_MAE_{n}", "")
            for n in target_names:
                _row[f"val_temporal_diff_MAE_{n}"] = val_metrics.get(f"temporal_diff_MAE_{n}", "")
            for _nc_key, _nc_m in _nc_out.items():
                _nc_row = {
                    "epoch": epoch, "split": "val", "node_count": _nc_key,
                    "MAE_x_abs":    _nc_m["MAE"].get("x_abs", ""),
                    "MAE_z_abs":    _nc_m["MAE"].get("z_abs", ""),
                    "MAE_tension":  _nc_m["MAE"].get("tension", ""),
                    "RMSE_x_abs":   _nc_m["RMSE"].get("x_abs", ""),
                    "RMSE_z_abs":   _nc_m["RMSE"].get("z_abs", ""),
                    "RMSE_tension": _nc_m["RMSE"].get("tension", ""),
                    "R2_x_abs":     _nc_m["R2"].get("x_abs", ""),
                    "R2_z_abs":     _nc_m["R2"].get("z_abs", ""),
                    "R2_tension":   _nc_m["R2"].get("tension", ""),
                    "global_R2_x_abs":    _nc_m["global_R2"].get("x_abs", ""),
                    "global_R2_z_abs":    _nc_m["global_R2"].get("z_abs", ""),
                    "global_R2_tension":  _nc_m["global_R2"].get("tension", ""),
                    "MAPE_tension":               _nc_m["MAPE"].get("tension", ""),
                    "peak_tension_MAE":           _nc_m.get("peak_tension_MAE", ""),
                    "peak_tension_bias":          _nc_m.get("peak_tension_bias", ""),
                    "peak_tension_underpred_rate":_nc_m.get("peak_tension_underpred_rate", ""),
                    "peak_tension_MAPE":          _nc_m.get("peak_tension_MAPE", ""),
                    "peak_tension_p95_abs_error": _nc_m.get("peak_tension_p95_abs_error", ""),
                    "peak_tension_max_abs_error": _nc_m.get("peak_tension_max_abs_error", ""),
                    "temporal_diff_MAE_x_abs":    _nc_m.get("temporal_diff_MAE_x_abs", ""),
                    "temporal_diff_MAE_z_abs":    _nc_m.get("temporal_diff_MAE_z_abs", ""),
                    "temporal_diff_MAE_tension":  _nc_m.get("temporal_diff_MAE_tension", ""),
                }
                _nc_writer.writerow(_nc_row)
            _nc_file.flush()

            if getattr(cfg, "select_on_r2_tension", False):
                selection_score = -val_metrics["global_R2"]["tension"]
            elif cfg.use_composite_checkpoint_score:
                _norm_peak  = val_metrics.get("peak_tension_MAE", 0.0) / max(
                    val_metrics["MAE"].get("tension", 1.0), 1.0)
                _norm_under = val_metrics.get("peak_tension_underpred_rate", 0.0) / 100.0
                selection_score = (
                    val_loss
                    + cfg.checkpoint_alpha * _norm_peak
                    + cfg.checkpoint_beta  * _norm_under
                )
            else:
                selection_score = val_loss
            _row["selection_score"] = selection_score
            _csv_writer.writerow(_row)
            _csv_file.flush()

            improved = (best_selection_score - selection_score) > cfg.min_delta
            if improved:
                best_selection_score       = selection_score
                best_val_loss_at_selection = val_loss
                best_state = copy.deepcopy(model.state_dict())
                _ckpt_dict = {
                    "model_state_dict":          model.state_dict(),
                    "optimizer_state_dict":      optimizer.state_dict(),
                    "scheduler_state_dict":      scheduler.state_dict() if scheduler else None,
                    "scaler_state_dict":         scaler.state_dict(),
                    "epoch":                     epoch,
                    "best_selection_score":       best_selection_score,
                    "best_val_loss_at_selection": best_val_loss_at_selection,
                    "best_val_loss":              best_val_loss_at_selection,
                    "cfg":                        _dc.asdict(cfg),
                }
                torch.save(_ckpt_dict, best_model_path)
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            # -- periodic checkpoint (save_every_n_epochs) ---------------
            if cfg.save_every_n_epochs > 0 and epoch % cfg.save_every_n_epochs == 0:
                _periodic_path = checkpoint_dir / f"checkpoint_epoch{epoch:04d}.pt"
                _pckpt = {
                    "model_state_dict":     model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
                    "scaler_state_dict":    scaler.state_dict(),
                    "epoch":                epoch,
                    "best_selection_score":       best_selection_score,
                    "best_val_loss_at_selection": best_val_loss_at_selection,
                    "best_val_loss":              best_val_loss_at_selection,
                    "cfg":                  _dc.asdict(cfg),
                }
                torch.save(_pckpt, _periodic_path)
                print(f"  Periodic checkpoint saved: {_periodic_path}")

            if epochs_without_improvement >= cfg.early_stopping_patience:
                print(f"Early stopping triggered at epoch {epoch}.")
                break

        else:
            history["val_loss"].append(None)
            history["val_metrics"].append(None)
            # Write base row with empty val fields for skipped epochs
            _csv_writer.writerow(_row)
            _csv_file.flush()
            print(
                f"Epoch {epoch:03d} | "
                f"train_loss={train_total_loss:.6f} (data={train_data_loss:.6f}) | "
                f"grad_norm={grad_norm:.4f} | amp_scale={amp_scale:.0f} | val: skipped"
            )

    _csv_file.close()
    _nc_file.close()

    # export CSV -> XLSX for direct cluster inspection (requires openpyxl)
    try:
        import pandas as _pd
        _pd.read_csv(history_path).to_excel(history_path.with_suffix('.xlsx'), index=False)
        _pd.read_csv(_nc_path).to_excel(_nc_path.with_suffix('.xlsx'), index=False)
        print(f"[xlsx] exported {history_path.with_suffix('.xlsx')} and {_nc_path.with_suffix('.xlsx')}")
    except Exception as _xlsx_err:
        print(f"[warn] XLSX export failed: {_xlsx_err}")

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, str(best_model_path)

In [17]:
# -------------------------------------------------------------
# test evaluation
# -------------------------------------------------------------


@torch.no_grad()
def test_model(
    model,
    test_loader,
    criterion,
    device,
    target_norm: FeatureStandardizer,
    target_names: List[str],
    cfg=None,
):
    _nc_out_test = {}
    test_loss, test_metrics = evaluate(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device,
        target_norm=target_norm,
        target_names=target_names,
        per_nc_metrics=_nc_out_test,
        cfg=cfg,
    )

    print("\n----- TEST RESULTS -----")
    print(f"test_loss = {test_loss:.6f}")

    for metric_name in ["MAE", "RMSE", "R2"]:
        for k, v in test_metrics[metric_name].items():
            print(f"test_{metric_name}_{k} = {v:.6f}")

    if "tension" in test_metrics["MAPE"]:
        print(f"test_MAPE_tension = {test_metrics['MAPE']['tension']:.6f}")

    for _k in ("peak_tension_MAE", "peak_tension_RMSE",
               "peak_tension_bias", "peak_tension_underpred_rate"):
        if _k in test_metrics:
            print(f"test_{_k} = {test_metrics[_k]:.6f}")

    for _h in (1, 10, 25, 50):
        _k = f"h{_h:02d}_MAE"
        if _k in test_metrics:
            print(f"test_{_k} = {test_metrics[_k]:.6f}")

    # New per-target horizon metrics
    for _tname in ["x_abs", "z_abs", "tension"]:
        for _h in (1, 10, 25, 50):
            _k = f"h{_h:02d}_MAE_{_tname}"
            if _k in test_metrics:
                print(f"test_{_k} = {test_metrics[_k]:.6f}")

    # Global R2
    if "global_R2" in test_metrics:
        for _k, _v in test_metrics["global_R2"].items():
            print(f"test_global_R2_{_k} = {_v:.6f}")

    # Additional peak tension diagnostics
    for _k in ("peak_tension_MAPE", "peak_tension_p95_abs_error",
               "peak_tension_max_abs_error"):
        if _k in test_metrics:
            print(f"test_{_k} = {test_metrics[_k]:.6f}")

    # Temporal smoothness
    for _tname in ["x_abs", "z_abs", "tension"]:
        _k = f"temporal_diff_MAE_{_tname}"
        if _k in test_metrics:
            print(f"test_{_k} = {test_metrics[_k]:.6f}")

    # -- persist test metrics to CSV (+ XLSX), mirroring the train/val files --
    if cfg is not None and getattr(cfg, "checkpoint_dir", None):
        from pathlib import Path as _Path
        import csv as _csv_t
        _out_dir = _Path(cfg.checkpoint_dir)
        _out_dir.mkdir(parents=True, exist_ok=True)
        _tn = list(target_names)
        _g = test_metrics.get("global_R2", {})
        # aggregate: tidy (metric, value), full val-style set with a test_ prefix
        _rows = [
            ("test_loss", test_loss),
            ("test_data_loss_norm", test_metrics.get("val_data_loss_norm", "")),
            ("test_phys_loss", test_metrics.get("val_phys_loss", "")),
        ]
        for _n in _tn: _rows.append((f"test_MAE_{_n}", test_metrics["MAE"].get(_n, "")))
        for _n in _tn: _rows.append((f"test_RMSE_{_n}", test_metrics["RMSE"].get(_n, "")))
        for _n in _tn: _rows.append((f"test_R2_{_n}", test_metrics["R2"].get(_n, "")))
        for _n in _tn: _rows.append((f"test_global_R2_{_n}", _g.get(_n, "")))
        _rows.append(("test_MAPE_tension", test_metrics["MAPE"].get("tension", "")))
        _rows.append(("test_MAPE_tension_valid_frac", test_metrics["MAPE"].get("tension_valid_frac", "")))
        for _k in ("peak_tension_MAE", "peak_tension_RMSE", "peak_tension_bias",
                   "peak_tension_underpred_rate", "peak_tension_MAPE",
                   "peak_tension_p95_abs_error", "peak_tension_max_abs_error"):
            _rows.append((f"test_{_k}", test_metrics.get(_k, "")))
        for _k in ("h01_MAE", "h10_MAE", "h25_MAE", "h50_MAE"):
            _rows.append((f"test_{_k}", test_metrics.get(_k, "")))
        for _h in (1, 10, 25, 50):
            for _n in _tn:
                _rows.append((f"test_h{_h:02d}_MAE_{_n}", test_metrics.get(f"h{_h:02d}_MAE_{_n}", "")))
        for _n in _tn:
            _rows.append((f"test_temporal_diff_MAE_{_n}", test_metrics.get(f"temporal_diff_MAE_{_n}", "")))
        _agg_csv = _out_dir / "test_metrics.csv"
        with open(_agg_csv, "w", newline="", encoding="utf-8") as _f:
            _w = _csv_t.writer(_f); _w.writerow(["metric", "value"]); _w.writerows(_rows)
        # per-node-count: same columns as metrics_by_node_count.csv, split=test
        _nc_fields = [
            "split", "node_count",
            "MAE_x_abs", "MAE_z_abs", "MAE_tension",
            "RMSE_x_abs", "RMSE_z_abs", "RMSE_tension",
            "R2_x_abs", "R2_z_abs", "R2_tension",
            "global_R2_x_abs", "global_R2_z_abs", "global_R2_tension",
            "MAPE_tension",
            "peak_tension_MAE", "peak_tension_bias",
            "peak_tension_underpred_rate", "peak_tension_MAPE",
            "peak_tension_p95_abs_error", "peak_tension_max_abs_error",
            "temporal_diff_MAE_x_abs", "temporal_diff_MAE_z_abs",
            "temporal_diff_MAE_tension",
        ]
        _nc_csv = _out_dir / "test_metrics_by_node_count.csv"
        with open(_nc_csv, "w", newline="", encoding="utf-8") as _f:
            _w = _csv_t.DictWriter(_f, fieldnames=_nc_fields, extrasaction="ignore")
            _w.writeheader()
            for _nc_key in sorted(_nc_out_test.keys()):
                _m = _nc_out_test[_nc_key]
                _w.writerow({
                    "split": "test", "node_count": _nc_key,
                    "MAE_x_abs":   _m["MAE"].get("x_abs", ""),
                    "MAE_z_abs":   _m["MAE"].get("z_abs", ""),
                    "MAE_tension": _m["MAE"].get("tension", ""),
                    "RMSE_x_abs":   _m["RMSE"].get("x_abs", ""),
                    "RMSE_z_abs":   _m["RMSE"].get("z_abs", ""),
                    "RMSE_tension": _m["RMSE"].get("tension", ""),
                    "R2_x_abs":   _m["R2"].get("x_abs", ""),
                    "R2_z_abs":   _m["R2"].get("z_abs", ""),
                    "R2_tension": _m["R2"].get("tension", ""),
                    "global_R2_x_abs":   _m["global_R2"].get("x_abs", ""),
                    "global_R2_z_abs":   _m["global_R2"].get("z_abs", ""),
                    "global_R2_tension": _m["global_R2"].get("tension", ""),
                    "MAPE_tension": _m["MAPE"].get("tension", ""),
                    "peak_tension_MAE":  _m.get("peak_tension_MAE", ""),
                    "peak_tension_bias": _m.get("peak_tension_bias", ""),
                    "peak_tension_underpred_rate": _m.get("peak_tension_underpred_rate", ""),
                    "peak_tension_MAPE":          _m.get("peak_tension_MAPE", ""),
                    "peak_tension_p95_abs_error": _m.get("peak_tension_p95_abs_error", ""),
                    "peak_tension_max_abs_error": _m.get("peak_tension_max_abs_error", ""),
                    "temporal_diff_MAE_x_abs":   _m.get("temporal_diff_MAE_x_abs", ""),
                    "temporal_diff_MAE_z_abs":   _m.get("temporal_diff_MAE_z_abs", ""),
                    "temporal_diff_MAE_tension": _m.get("temporal_diff_MAE_tension", ""),
                })
        try:
            import pandas as _pd_t
            _pd_t.read_csv(_agg_csv).to_excel(_agg_csv.with_suffix(".xlsx"), index=False)
            _pd_t.read_csv(_nc_csv).to_excel(_nc_csv.with_suffix(".xlsx"), index=False)
            print(f"[test-metrics] wrote {_agg_csv.name}, {_nc_csv.name} (+ xlsx) to {_out_dir}")
        except Exception as _xe:
            print(f"[test-metrics] CSV written; XLSX export failed: {_xe}")

    return {
        "test_loss": test_loss,
        "test_metrics": test_metrics,
    }

In [18]:
def run_multi_seed_test_experiment(
    load_case_datasets: Dict[int, MooringSequenceDatasetPositionTension],
    cfg: TrainingConfig,
    device: torch.device,
):
    first_key = sorted(load_case_datasets.keys())[0]
    example_dataset = load_case_datasets[first_key]

    # Fixed split for all seeds
    train_loader, val_loader, test_loader, norms, split_indices = build_dataloaders(
        load_case_datasets=load_case_datasets,
        cfg=cfg,
    )

    all_results = []

    for run_seed in cfg.seed_list:
        print("\n" + "=" * 70)
        print(f"STARTING RUN FOR SEED {run_seed}")
        print("=" * 70)

        set_seed(run_seed)

        model = build_model_from_config(
            example_dataset=example_dataset,
            cfg=cfg,
            device=device,
        )

        criterion = SequenceSmoothL1Loss(beta=1.0)
        optimizer, scheduler = build_optimizer_and_scheduler(model, cfg)

        model, history, model_path = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            criterion=criterion,
            device=device,
            target_norm=norms["target_norm"],
            target_names=example_dataset.target_feature_names,
            cfg=cfg,
            run_seed=run_seed,
        )

        test_results = test_model(
            model=model,
            test_loader=test_loader,
            criterion=criterion,
            device=device,
            target_norm=norms["target_norm"],
            target_names=example_dataset.target_feature_names,
        )

        run_result = {
            "seed": run_seed,
            "model_path": model_path,
            "history": history,
            "test_loss": test_results["test_loss"],
            "test_metrics": test_results["test_metrics"],
        }
        all_results.append(run_result)

    # Aggregate test loss across seeds
    test_losses = [r["test_loss"] for r in all_results]
    mean_test_loss = float(np.mean(test_losses))
    std_test_loss = float(np.std(test_losses))

    # Aggregate test across seeds
    target_names = example_dataset.target_feature_names
    metric_names_all_targets = ["MAE", "RMSE", "R2"]
    mean_test_metrics = {metric: {} for metric in metric_names_all_targets}
    std_test_metrics = {metric: {} for metric in metric_names_all_targets}
    mean_test_metrics["MAPE"] = {}
    std_test_metrics["MAPE"] = {}

    for metric in metric_names_all_targets:
        for name in target_names:
            vals = [r["test_metrics"][metric][name] for r in all_results]
            mean_test_metrics[metric][name] = float(np.mean(vals))
            std_test_metrics[metric][name] = float(np.std(vals))

    if "tension" in target_names:
        vals = [r["test_metrics"]["MAPE"].get("tension", float("nan")) for r in all_results]
        mean_test_metrics["MAPE"]["tension"] = float(np.nanmean(vals))
        std_test_metrics["MAPE"]["tension"] = float(np.nanstd(vals))


    print("\n" + "=" * 70)
    print("MULTI-SEED TEST SUMMARY")
    print("=" * 70)
    print(f"Seeds used: {list(cfg.seed_list)}")
    print(f"Mean test loss across seeds = {mean_test_loss:.6f}")
    print(f"Std  test loss across seeds = {std_test_loss:.6f}")

    for metric in ["MAE", "RMSE", "R2"]:
        print(f"\n{metric}:")
        for name in target_names:
            print(
                f"  {name}: mean = {mean_test_metrics[metric][name]:.6f}, "
                f"std = {std_test_metrics[metric][name]:.6f}"
            )

    if "tension" in mean_test_metrics["MAPE"]:
        print("\nMAPE:")
        print(
            f"  tension: mean = {mean_test_metrics['MAPE']['tension']:.6f}, "
            f"std = {std_test_metrics['MAPE']['tension']:.6f}"
        )

    summary = {
        "all_results": all_results,
        "mean_test_loss": mean_test_loss,
        "std_test_loss": std_test_loss,
        "mean_test_metrics": mean_test_metrics,
        "std_test_metrics": std_test_metrics,
    }
    return summary

In [19]:
# -------------------------------------------------------------
# real data loader
# -------------------------------------------------------------

import os as _os

DATA_ROOT = Path(r"C:\Users\thano\Desktop\data")  # local path; not needed in cache-only runs
CACHE_DIR = Path(
    _os.environ.get(
        "MOORING_CACHE_DIR",
        "/scratch/tkonstantaras/mooring_data/cache_npy",
    )
)
AVAILABLE_LOC_IDS = [1, 3, 4, 5, 6, 7, 8, 9, 10, 11]
N_CASES = 300

# Corrupt FE simulation cases: max(tension) > 5 MN indicating numerical
# divergence in the FE solver. Excluded from dataset construction so they
# do not contaminate the FeatureStandardizer or any training/val windows.
CORRUPT_CASES = frozenset({
    # Excluded as FE-solver divergence: non-finite (NaN/inf) tension OR peak |T| > 1.5 MN.
    # 1.5 MN cut: (8,83)=1.42 MN (8.4x its loc p99) KEPT as a plausible snap load (up to ~9x cyclic max);
    (1, 63), (1, 202), (1, 215),
    (4, 45), (4, 133), (4, 164), (4, 270),
    (5, 31), (5, 35), (5, 46), (5, 108), (5, 194), (5, 197), (5, 202), (5, 262), (5, 264),
    (6, 22), (6, 131), (6, 134),
    (7, 16), (7, 32), (7, 83), (7, 132), (7, 137), (7, 174), (7, 212), (7, 214), (7, 220), (7, 221), (7, 225), (7, 229),
    (8, 37), (8, 94), (8, 136), (8, 143), (8, 247),
    (9, 43), (9, 167), (9, 193),
    (10, 22), (10, 56), (10, 77), (10, 187), (10, 188), (10, 224), (10, 234), (10, 239), (10, 275), (10, 278),
    (11, 3), (11, 22), (11, 61), (11, 89),
})
N_NODES_FULL = 21
CSV_SEP = ";"

# Actual input CSV columns:
# SN is the 1-indexed case number. h0 is the water depth for that case.
ENV_COL_NAMES = [
    "h0", "Hs", "Tp",
    "d1", "v1",
    "d2", "v2",
    "d3", "v3",
    "d4", "v4",
    "d5", "v5",
]
DEPTH_COL_NAMES = ["d1", "d2", "d3", "d4", "d5"]
REQUIRED_ENV_COLUMNS = ["SN", *ENV_COL_NAMES]

def _loc_folder(loc_id: int) -> Path:
    return DATA_ROOT / f"batchRieke_loc{loc_id:02d}"

def _case_dat_path(loc_id: int, case_id: int) -> Path:
    return _loc_folder(loc_id) / f"case_{case_id:04d}" / "gnl_data1.dat"

def inspect_env_csv(loc_id: int) -> None:
    """Print column names and first row of the env CSV for loc_id."""
    input_dir = _loc_folder(loc_id) / "input"
    csvs = sorted(input_dir.glob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSV found in {input_dir}")
    df = pd.read_csv(csvs[0], sep=CSV_SEP)
    print(f"Loc {loc_id:02d} env CSV: {csvs[0].name}")
    print(f"  Columns ({len(df.columns)}): {list(df.columns)}")
    print(f"  First row: {df.iloc[0].to_dict()}")
    print(f"  Shape: {df.shape}")

def load_env_csv(loc_id: int) -> pd.DataFrame:
    """Return the env DataFrame for a location, using the actual semicolon CSV schema."""
    input_dir = _loc_folder(loc_id) / "input"
    csvs = sorted(input_dir.glob("*.csv"))
    if len(csvs) != 1:
        raise FileNotFoundError(f"Expected 1 CSV in {input_dir}, found: {[p.name for p in csvs]}")

    df = pd.read_csv(csvs[0], sep=CSV_SEP)
    missing = [c for c in REQUIRED_ENV_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(
            f"CSV {csvs[0].name} is missing columns {missing}. "
            f"Found columns: {list(df.columns)}"
        )
    if len(df) < N_CASES:
        raise ValueError(f"Expected at least {N_CASES} cases in {csvs[0].name}, found {len(df)}.")
    return df

def case_env_row(env_df: pd.DataFrame, case_id: int) -> pd.Series:
    """Return the CSV row whose SN equals the 1-indexed case number."""
    matches = env_df.loc[env_df["SN"] == case_id]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one row with SN={case_id}, found {len(matches)}.")
    return matches.iloc[0]

def case_water_depth(env_df: pd.DataFrame, case_id: int) -> float:
    """Return h0 [m] from the CSV for one case."""
    return float(case_env_row(env_df, case_id)["h0"])

def case_current_depths(env_df: pd.DataFrame, case_id: int) -> list:
    """Return d1..d5 [m] from the CSV for one case."""
    row = case_env_row(env_df, case_id)
    return [float(row[col]) for col in DEPTH_COL_NAMES]

def case_env_tensor(env_df: pd.DataFrame, case_id: int, T: int) -> torch.Tensor:
    """
    Return a [T, 13] float32 tensor with constant env features for one case.

    Parameters
    ----------
    env_df  : DataFrame returned by load_env_csv().
    case_id : 1-indexed case number; matched to the SN column.
    T       : number of time steps (tensor rows will all be identical).
    """
    row_values = case_env_row(env_df, case_id)[ENV_COL_NAMES].to_numpy(dtype=np.float32)
    row_t = torch.tensor(row_values, dtype=torch.float32)   # [13]
    return row_t.unsqueeze(0).expand(T, -1).contiguous()    # [T, 13]

def load_gnl_dat(dat_path: Path, n_nodes: int = 21):
    """
    Load one gnl_data1.dat file.

    Exact file layout:
        - No header
        - Comma-separated
        - 87 columns total for 21 nodes
        - Column 0: time / timestep
        - Columns 1 and 2: unused, drop them
        - Then repeated per node:
            reference, x_abs, z_abs, tension

    Raw expected number of columns:
        3 + 4*n_nodes

    After dropping columns 1 and 2:
        1 + 4*n_nodes

    Returns:
        time:    [T]
        x_abs:   [T, n_nodes]
        z_abs:   [T, n_nodes]
        tension: [T, n_nodes]
    """
    dat_path = Path(dat_path)
    if not dat_path.exists():
        raise FileNotFoundError(f"Missing .dat file: {dat_path}")

    raw_df = pd.read_csv(dat_path, sep=",", header=None)

    raw_expected_cols = 3 + 4 * n_nodes
    if raw_df.shape[1] != raw_expected_cols:
        raise ValueError(
            f"{dat_path} has {raw_df.shape[1]} raw columns, but expected "
            f"{raw_expected_cols} for n_nodes={n_nodes}. "
            "Expected layout: time, unused1, unused2, then "
            "[reference, x_abs, z_abs, tension] repeated for each node."
        )

    df = raw_df.drop(columns=[1, 2])

    expected_cols_after_drop = 1 + 4 * n_nodes
    if df.shape[1] != expected_cols_after_drop:
        raise ValueError(
            f"After dropping columns 1 and 2, {dat_path} has {df.shape[1]} columns, "
            f"but expected {expected_cols_after_drop}."
        )

    time = df.iloc[:, 0].to_numpy(dtype=np.float32)
    node_values = df.iloc[:, 1:].to_numpy(dtype=np.float32)

    if node_values.shape[1] % 4 != 0:
        raise ValueError(
            f"After dropping columns 1 and 2, node data has "
            f"{node_values.shape[1]} columns, which is not divisible by 4."
        )

    inferred_nodes = node_values.shape[1] // 4
    if inferred_nodes != n_nodes:
        raise ValueError(
            f"Inferred {inferred_nodes} nodes from file, but n_nodes={n_nodes}. "
            "Check N_FULL or the file format."
        )

    node_data = node_values.reshape(node_values.shape[0], inferred_nodes, 4)

    reference = node_data[:, :, 0]  # arc-length s [m] per node
    x_abs     = node_data[:, :, 1]
    z_abs     = node_data[:, :, 2]
    tension   = node_data[:, :, 3]

    return time, x_abs, z_abs, tension, reference

def preprocess_all_data(cache_dir: Path, loc_ids, case_ids):
    """
    One-time preprocessing: convert each .dat file to four uncompressed .npy files
    (x_abs, z_abs, tension, reference) plus one env.npy per case.

    Uses uncompressed .npy so np.load(..., mmap_mode='r') works.
    Skips cases whose output directory already exists.
    """
    for loc_id in loc_ids:
        env_df = load_env_csv(loc_id)
        for case_id in case_ids:
            if (loc_id, case_id) in corrupt_cases:
                continue
            case_dir = cache_dir / f"loc{loc_id:02d}" / f"case_{case_id:04d}"
            if (case_dir / "x_abs.npy").exists():
                continue
            case_dir.mkdir(parents=True, exist_ok=True)
            _time, x_abs, z_abs, tension, reference = load_gnl_dat(
                _case_dat_path(loc_id, case_id), n_nodes=N_NODES_FULL
            )
            env_row = case_env_row(env_df, case_id)[ENV_COL_NAMES].to_numpy(np.float32)
            np.save(case_dir / "x_abs.npy",     x_abs.astype(np.float32))     # [T, 21]
            np.save(case_dir / "z_abs.npy",     z_abs.astype(np.float32))
            np.save(case_dir / "tension.npy",   tension.astype(np.float32))
            np.save(case_dir / "reference.npy", reference.astype(np.float32))
            np.save(case_dir / "env.npy",       env_row)                        # [13] env row
        print(f"loc {loc_id:02d}: done")
    print("Preprocessing complete.")


In [ ]:
# preprocess_all_data(
#     cache_dir=CACHE_DIR,
#     loc_ids=AVAILABLE_LOC_IDS,
#     case_ids=list(range(1, N_CASES + 1)),
# )

loc 01: done
loc 03: done
loc 04: done
loc 05: done
loc 06: done


C:\Users\thano\AppData\Local\Temp\ipykernel_24952\4084240722.py:120: DtypeWarning: Columns (0: 86) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(dat_path, sep=",", header=None)


loc 07: done
loc 08: done
loc 09: done
loc 10: done


C:\Users\thano\AppData\Local\Temp\ipykernel_24952\4084240722.py:120: DtypeWarning: Columns (0: 82) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(dat_path, sep=",", header=None)


loc 11: done
Preprocessing complete.


In [ ]:
# -------------------------------------------------------------
# build load_case_datasets from .npy cache (lazy mmap)
# -------------------------------------------------------------
#
# This debug cell intentionally loads a small real-data subset so the full
# pipeline can be verified quickly. Increase DEBUG_* values later for the
# full experiment.

DEBUG_LOC_IDS = [4]
DEBUG_CASE_IDS = [1]
DEBUG_NODE_COUNTS = [4, 8, 12, 21]
DEBUG_MAX_TIMESTEPS = 1000       # use None to load all ~13,800 time steps

HISTORY_LEN = 250
FUTURE_LEN = 50


# ------------------------------------------------------------------
# Full-scale lazy dataset builder (uses .npy cache via mmap)
# ------------------------------------------------------------------

def build_lazy_load_case_datasets(
    loc_ids=AVAILABLE_LOC_IDS,
    case_ids=range(1, N_CASES + 1),
    node_counts=(4, 5, 6, 7, 8, 10, 12, 15, 18, 21),
    cache_dir=CACHE_DIR,
    history_len=HISTORY_LEN,
    future_len=FUTURE_LEN,
    contact_tol=1e-6,
    max_time_steps=None,
    corrupt_cases=frozenset(),
):
    """
    Build one lazy MooringSequenceDatasetPositionTension per (loc_id, case_id, target_N).

    Data is read from the pre-processed .npy cache (see preprocess_all_data).
    Each dataset object stores only a directory path -- no time-series data in RAM.
    File handles are opened per DataLoader worker on first access.

    Parameters
    ----------
    max_time_steps : int, optional
        Limit exposed time steps for debug runs. None = use all steps.
    """
    bad_locs = sorted(set(loc_ids) - set(AVAILABLE_LOC_IDS))
    if bad_locs:
        raise ValueError(f"Unavailable locations requested: {bad_locs}.")

    datasets = {}

    for loc_id in loc_ids:
        print(f"\n--- Location {loc_id:02d} ---")

        for case_id in case_ids:
            if (loc_id, case_id) in corrupt_cases:
                continue
            case_dir = cache_dir / f"loc{loc_id:02d}" / f"case_{case_id:04d}"
            if not (case_dir / "x_abs.npy").exists():
                raise FileNotFoundError(
                    f"Cache missing for loc {loc_id}, case {case_id}. "
                    "Run preprocess_all_data() first."
                )

            # Read only t=0 row for graph construction -- 21 floats, negligible
            x0 = torch.tensor(np.load(case_dir / "x_abs.npy",     mmap_mode="r")[0], dtype=torch.float32)
            z0 = torch.tensor(np.load(case_dir / "z_abs.npy",     mmap_mode="r")[0], dtype=torch.float32)
            t0 = torch.tensor(np.load(case_dir / "tension.npy",   mmap_mode="r")[0], dtype=torch.float32)
            s0 = torch.tensor(np.load(case_dir / "reference.npy", mmap_mode="r")[0], dtype=torch.float32)

            # env.npy layout: [h0, Hs, Tp, d1, v1, d2, v2, d3, v3, d4, v4, d5, v5]
            env_row        = np.load(case_dir / "env.npy")
            water_depth    = float(env_row[0])                               # h0
            current_depths = [float(env_row[i]) for i in (3, 5, 7, 9, 11)]  # d1..d5

            for target_N in node_counts:
                graph = build_mooring_graph(
                    x0_nodes=resample_fe_output(x0.unsqueeze(0), target_N).squeeze(0),
                    z0_nodes=resample_fe_output(z0.unsqueeze(0), target_N).squeeze(0),
                    tension0_nodes=resample_fe_output(t0.unsqueeze(0), target_N).squeeze(0),
                    s0_nodes=resample_fe_output(s0.unsqueeze(0), target_N).squeeze(0),
                    water_depth=water_depth,
                    depths_current=current_depths,
                )
                graph.batch_location_id = loc_id
                graph.case_id           = case_id

                datasets[(loc_id, case_id, target_N)] = MooringSequenceDatasetPositionTension(
                    graph_data=graph,
                    case_dir=case_dir,
                    target_N=target_N,
                    history_len=history_len,
                    future_len=future_len,
                    contact_tol=contact_tol,
                    max_time_steps=max_time_steps,
                )

        print(f"  loc {loc_id:02d}: {len(list(case_ids))} cases x {len(list(node_counts))} node counts built")

    print(f"\nTotal lazy datasets: {len(datasets):,}")
    return datasets


# Stage C: all locs (train + held-out), cases 1-300, all node_counts, full time series
load_case_datasets = build_lazy_load_case_datasets(
    loc_ids=[1, 3, 4, 5, 6, 7, 8, 9, 10, 11],
    case_ids=range(1, 301),
    node_counts=(4, 5, 6, 7, 8, 10, 12, 15, 18, 21),
    cache_dir=CACHE_DIR,
    history_len=HISTORY_LEN,
    future_len=FUTURE_LEN,
    max_time_steps=None,
    corrupt_cases=CORRUPT_CASES,
)

# ------------------------------------------------------------------
# Consistency checks
# ------------------------------------------------------------------
assert len(load_case_datasets) > 0, "load_case_datasets is empty."

first_key = sorted(load_case_datasets.keys())[0]
example_ds = load_case_datasets[first_key]

ref_dyn_node  = example_ds.n_dynamic_node_features
ref_dyn_edge  = example_ds.n_dynamic_edge_features
ref_target    = example_ds.n_targets
ref_hist      = example_ds.history_len
ref_fut       = example_ds.future_len
ref_node_feat = example_ds.graph_data.x.shape[1]
assert ref_node_feat == 14, f"Expected 14 static node features, got {ref_node_feat}"
ref_edge_feat = example_ds.graph_data.edge_attr.shape[1]
assert ref_edge_feat == 10, f"Expected 10 static edge features, got {ref_edge_feat}"

for (lc_id, case_id, target_N), ds in load_case_datasets.items():
    tag = f"({lc_id},{case_id},{target_N})"
    assert ds.n_dynamic_node_features == ref_dyn_node, f"{tag}: dyn node feat mismatch"
    assert ds.n_dynamic_edge_features == ref_dyn_edge, f"{tag}: dyn edge feat mismatch"
    assert ds.n_targets               == ref_target,   f"{tag}: target dim mismatch"
    assert ds.history_len == ref_hist,                 f"{tag}: history_len mismatch"
    assert ds.future_len  == ref_fut,                  f"{tag}: future_len mismatch"
    assert ds.graph_data.x.shape[1]        == ref_node_feat, f"{tag}: static node feat mismatch"
    assert ds.graph_data.edge_attr.shape[1] == ref_edge_feat, f"{tag}: static edge feat mismatch"

print_dataset_summary(example_ds)


# ------------------------------------------------------------------
# OBSOLETE: build_real_load_case_datasets
# Kept for reference only. Incompatible with the lazy dataset class:
# passes x_abs/z_abs/tension tensors instead of a case_dir Path, and
# eagerly loads the full time series into RAM.
# Use build_lazy_load_case_datasets for all new code.
# ------------------------------------------------------------------

def build_real_load_case_datasets(
    loc_ids=AVAILABLE_LOC_IDS,
    case_ids=range(1, N_CASES + 1),
    node_counts=(4, 5, 6, 7, 8, 10, 12, 15, 18, 21),
    max_time_steps=None,
):
    """
    Build one dataset per (loc_id, case_id, target_N).

    Cases are separate sea states and are not concatenated along time.
    """
    bad_locs = sorted(set(loc_ids) - set(AVAILABLE_LOC_IDS))
    if bad_locs:
        raise ValueError(f"Unavailable locations requested: {bad_locs}. Use {AVAILABLE_LOC_IDS}.")

    load_case_datasets = {}

    for loc_id in loc_ids:
        print(f"\n--- Location {loc_id:02d} ---")
        env_df = load_env_csv(loc_id)

        for case_id in case_ids:
            dat_path = _case_dat_path(loc_id, case_id)
            time, x_abs_full_np, z_abs_full_np, tension_full_np, reference_full_np = load_gnl_dat(
                dat_path, n_nodes=N_NODES_FULL
            )

            if max_time_steps is not None:
                time = time[:max_time_steps]
                x_abs_full_np      = x_abs_full_np[:max_time_steps]
                z_abs_full_np      = z_abs_full_np[:max_time_steps]
                tension_full_np    = tension_full_np[:max_time_steps]
                reference_full_np  = reference_full_np[:max_time_steps]

            x_abs_full    = torch.tensor(x_abs_full_np,   dtype=torch.float32)
            z_abs_full    = torch.tensor(z_abs_full_np,   dtype=torch.float32)
            tension_full  = torch.tensor(tension_full_np, dtype=torch.float32)
            reference_full = torch.tensor(reference_full_np, dtype=torch.float32)

            T = x_abs_full.shape[0]
            env_t = case_env_tensor(env_df, case_id, T)
            water_depth = case_water_depth(env_df, case_id)
            current_depths = case_current_depths(env_df, case_id)

            x0_full       = x_abs_full[0]
            z0_full       = z_abs_full[0]
            tension0_full = tension_full[0]
            s0_full       = reference_full[0]

            for target_N in node_counts:
                x_abs   = resample_fe_output(x_abs_full, target_N)
                z_abs   = resample_fe_output(z_abs_full, target_N)
                tension = resample_fe_output(tension_full, target_N)
                x0_init = resample_fe_output(x0_full.unsqueeze(0), target_N).squeeze(0)
                z0_init = resample_fe_output(z0_full.unsqueeze(0), target_N).squeeze(0)
                t0_init = resample_fe_output(tension0_full.unsqueeze(0), target_N).squeeze(0)
                s0_init = resample_fe_output(s0_full.unsqueeze(0), target_N).squeeze(0)

                graph = build_mooring_graph(
                    x0_nodes=x0_init,
                    z0_nodes=z0_init,
                    tension0_nodes=t0_init,
                    s0_nodes=s0_init,
                    water_depth=water_depth,
                    depths_current=current_depths,
                )
                graph.batch_location_id = loc_id
                graph.case_id = case_id

                load_case_datasets[(loc_id, case_id, target_N)] = MooringSequenceDatasetPositionTension(
                    graph_data=graph,
                    x_abs=x_abs,
                    z_abs=z_abs,
                    tension=tension,
                    env_features=env_t,
                    history_len=HISTORY_LEN,
                    future_len=FUTURE_LEN,
                    contact_tol=1e-6,
                )

            print(f"  case {case_id:04d}: T={T}, h0={water_depth:.3f}, node_counts={list(node_counts)}")

    print(f"\nTotal datasets: {len(load_case_datasets):,}")
    print(f"  = {len(list(loc_ids))} locs x {len(list(case_ids))} cases x {len(list(node_counts))} node counts")
    return load_case_datasets



--- Location 04 ---
  loc 04: 1 cases x 4 node counts built

Total lazy datasets: 4
----- DATASET SUMMARY -----
Time steps : 1000
Nodes      : 4
History len: 250
Future len : 50
Windows    : 701
Dynamic input features : ['x_abs', 'z_abs', 'tension', 'contact_flag', 'penetration_depth', 'bed_reaction_z', 'h0', 'Hs', 'Tp', 'd1', 'v1', 'd2', 'v2', 'd3', 'v3', 'd4', 'v4', 'd5', 'v5']
Dynamic edge  features : ['edge_contact_fraction', 'edge_penetration_mean', 'edge_bed_reaction_mean', 'edge_touchdown_flag']
Target features        : ['x_abs', 'z_abs', 'tension']
x_seq   : (250, 4, 19)
edge_seq: (250, 6, 4)
y_seq   : (50, 4, 3)


In [ ]:
# -------------------------------------------------------------
# Stage C2c -- Kendall uncertainty weighting on ALL 4 physical terms, with the
# peak-UNDERprediction term reformulated as a tau=0.9 PINBALL (quantile) loss.
# C2a Kendall-weighted the one-sided relu hinge -> no noise floor -> sigma runaway
# (job 10198256). C2b held underpred FIXED -> the 3 two-sided sigma converged but
# the fixed safety term was swamped by the auto-scaled Kendall weights (job 10202988).
# The pinball loss is two-sided (penalises under 9x more than over) AND has a floor,
# so its sigma converges -> a clean 4-way importance ranking. Best checkpoint is
# selected on val_global_R2_tension (the Kendall val_loss is not a quality proxy).
# Same BE1 arch + full 300 cases/location, seed 42.
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

cfg = TrainingConfig(
    train_lc_ids=(1, 3, 4, 5, 6, 7, 8, 9),
    test_extra_lc_ids=(10, 11),
    node_counts=(4, 5, 6, 7, 8, 10, 12, 15, 18, 21),
    raw_block_len=500,
    max_train_windows=500_000,          # ~21 win / train dataset (was 1 if left at 20k)
    max_val_windows=10_000,
    max_test_windows=30_000,
    max_train_windows_per_epoch=30_000, # fresh resample each epoch
    n_fit_windows=5_000,
    num_epochs=100,                     # ceiling; early stopping fires first
    learning_rate=1e-3,
    weight_decay=1e-4,
    batch_size=32,
    grad_clip_max_norm=1.0,
    lstm_hidden_dim=256,        # both BBF and BE1
    num_lstm_layers=2,          # BE1 winner
    # --- Regularisation: dropout (C1 default = 0.1, the defended model) ---
    gat_dropout=0.1,
    lstm_dropout=0.1,
    head_dropout=0.1,
    # Dropout=0.15 overfitting check (C3): comment the 3 lines above, uncomment
    # the 3 below, AND switch checkpoint_dir below so the C1 model is not overwritten.
    # gat_dropout=0.15,
    # lstm_dropout=0.15,
    # head_dropout=0.15,
    use_scheduler=True,
    scheduler_factor=0.5,
    scheduler_patience=5,
    early_stopping_patience=50,  # C2c: +20 vs C2b so the run is long enough to SEE the pinball sigma plateau (does not change which ckpt is selected -- still best val_R2_tension)
    min_delta=1e-5,
    enable_bucketed_node_count_batching=True,
    drop_last_batch=False,
    num_workers=4,
    pin_memory=(device.type == "cuda"),
    use_amp=False,
    validate_every=1,
    save_every_n_epochs=5,
    debug_batch_shapes=False,
    seed_list=(42,),
    checkpoint_dir="./checkpoints_C2c_pinball",
    # checkpoint_dir="./checkpoints_C3_dropout015",  # use this when dropout=0.15 is active
    run_test_evaluation=True,           # Stage C final run
    compute_train_physical_metrics=True,
    max_train_metric_batches=50,
    use_physical_loss_terms=True,
    use_kendall_weighting=True,         # C2c: learnable log_sigma weight ALL 4 physical terms
    select_on_r2_tension=True,          # select best ckpt by max val_global_R2_tension (Kendall val_loss != quality)
    peak_pinball_tau=0.9,               # C2c: underpred term = tau=0.9 pinball (two-sided, has a noise floor)
    # --- Physical-loss weighting (C2c) ------------------------------------------
    # All 4 terms Kendall-weighted (learnable log_sigma): phys_mae, peak_tension,
    # peak_underprediction (now a two-sided pinball, NOT the one-sided relu hinge),
    # temporal. Their fixed w_* are all inert under Kendall and stay commented:
    # w_phys_mae=0.0025,
    # w_peak_tension=0.005,
    # w_peak_underprediction=0.02,
    # w_temporal_smoothness=0.0025,
    # Revert to C1 fixed-weight objective: use_kendall_weighting=False AND uncomment
    # all four w_* above (C1 uses the one-sided relu hinge for underprediction).
)

set_seed(cfg.seed_list[0])

t0 = time.time()
train_loader, val_loader, test_loader, norms, split_indices = build_dataloaders(
    load_case_datasets=load_case_datasets, cfg=cfg,
)
print(f"build_dataloaders took {time.time() - t0:.1f} s")

first_key = sorted(load_case_datasets.keys())[0]
example_dataset = load_case_datasets[first_key]

model = build_model_from_config(example_dataset=example_dataset, cfg=cfg, device=device)
criterion = SequenceSmoothL1Loss(beta=1.0)
optimizer, scheduler = build_optimizer_and_scheduler(model, cfg)

t0 = time.time()
model, history, model_path = train_model(
    model=model, train_loader=train_loader, val_loader=val_loader,
    optimizer=optimizer, scheduler=scheduler, criterion=criterion,
    device=device, target_norm=norms["target_norm"],
    target_names=example_dataset.target_feature_names,
    cfg=cfg, run_seed=cfg.seed_list[0],
)
print(f"training took {time.time() - t0:.1f} s")
if torch.cuda.is_available():
    print(f"Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

if cfg.run_test_evaluation:
    debug_test_results = test_model(
        model=model, test_loader=test_loader, criterion=criterion,
        device=device, target_norm=norms["target_norm"],
        target_names=example_dataset.target_feature_names, cfg=cfg,
    )
else:
    debug_test_results = None
    print("Test evaluation skipped.")

print("\nRun complete.")
print(f"Best checkpoint: {model_path}")
print(f"Final train loss: {history['train_loss'][-1]:.6f}")
print(f"Final val loss:   {history['val_loss'][-1]:.6f}" if history['val_loss'] else "Final val loss:   (no validation completed)")

Using device: cuda
----- SPLIT SUMMARY -----

TRAIN total windows: 40
  LC 04 N=04: 10 windows
  LC 04 N=08: 10 windows
  LC 04 N=12: 10 windows
  LC 04 N=21: 10 windows

VAL total windows: 20
  LC 04 N=04: 5 windows
  LC 04 N=08: 5 windows
  LC 04 N=12: 5 windows
  LC 04 N=21: 5 windows

TEST total windows: 4
  LC 04 N=04: 1 windows
  LC 04 N=08: 1 windows
  LC 04 N=12: 1 windows
  LC 04 N=21: 1 windows
build_dataloaders took 0.4 s
  [batch debug] step=0  x_seq=(2, 250, 4, 19)  node_count=4  n_graphs=2
  [batch debug] step=1  x_seq=(4, 250, 12, 19)  node_count=12  n_graphs=4
  [batch debug] step=2  x_seq=(2, 250, 21, 19)  node_count=21  n_graphs=2
Epoch 001 | train_loss=0.413510 | grad_norm=1.6553 | val_loss=0.302950 | val_MAE_x_abs=18.505461 | val_MAE_z_abs=8.383103 | val_MAE_tension=2008.833765 | val_RMSE_x_abs=22.425219 | val_RMSE_z_abs=11.659604 | val_RMSE_tension=2813.403650 | val_R2_x_abs=0.268132 | val_R2_z_abs=0.394153 | val_R2_tension=0.397147 | val_MAPE_tension=55.504746
  [

In [23]:
# -------------------------------------------------------------
# CPU/GPU device-isolation smoke test
# -------------------------------------------------------------

def _smoke_test_device_isolation(ds):
    s0 = ds[0]
    assert s0["x_seq"].device.type == "cpu", "x_seq must be on CPU after __getitem__"
    assert s0["graph"].x.device.type == "cpu", "graph.x must be on CPU after __getitem__"

    if torch.cuda.is_available():
        device = torch.device("cuda")
        moved = move_sample_to_device(s0, device)
        assert moved["x_seq"].device.type == "cuda", "moved x_seq must be on CUDA"
        assert moved["graph"].x.device.type == "cuda", "moved graph.x must be on CUDA"

        # Fetch again — dataset graph must still be on CPU
        s1 = ds[1]
        assert s1["x_seq"].device.type == "cpu", "x_seq must remain on CPU after GPU move"
        assert s1["graph"].x.device.type == "cpu", "graph.x must remain on CPU after GPU move"
        print("smoke test passed (CPU + CUDA checks)")
    else:
        print("smoke test passed (CPU-only: no CUDA available)")


# Run against the first available lazy dataset
_first_ds = next(iter(load_case_datasets.values()))
_smoke_test_device_isolation(_first_ds)

smoke test passed (CPU + CUDA checks)
